# SEM-CHRONO (nửa 1/2) — ChronoDeSmear Round 5 trên SVHalluc semantic (GSA / FGSA / CSMB)

**Tài khoản Kaggle #18.** GPU **T4 ×2**, bật **Internet** (Session options), rồi **Save Version → Save & Run All**. Thời gian ≈ 2–4 giờ (chưa đo). Khi xong, tải **`semchrono_s1.zip`** ở cell cuối và gửi cho Claude.

Notebook tự chạy thử 4 mẫu trước; nếu có lỗi thật (không tính clip quá dài), nó dừng ngay để không phí GPU. Job tự dừng an toàn trước hạn 12 giờ của Kaggle và giữ những gì đã chạy xong.

### Vì sao chạy
Phương pháp bạn của nhóm đang thử: **ChronoDeSmear (Round 5)** — trừ "tiếng vang nhân quả" và "nền tĩnh" khỏi các token hình/tiếng ở các tầng giữa, lấy phần chênh lệch với các lượt giả định (chỉ nền tĩnh + tiếng; chỉ hình) rồi bơm ngược vào token câu hỏi/đáp án. Trên bộ temporal 40 mẫu, bản tốt nhất (R5_C2) đạt 21/40 so với native 18/40 (p = 0,63): chưa có gain ổn định. Câu hỏi semantic (GSA, FGSA, CSMB) phụ thuộc chủ yếu vào việc tách điều THẤY khỏi điều NGHE, đúng phần mà phương pháp này có tín hiệu (CMTB). Notebook chạy **đúng mã của bạn bạn, không sửa phương pháp**, chỉ đổi đường dẫn manifest/cache/kết quả.

### Đo gì
Bộ semantic cân bằng, seed 31337: 6 ô (GSA, FGSA, CSMB) x (A = Yes, B = No), 10 mẫu mỗi ô = 60 mẫu, clip <= 40 s; notebook này chạy một nửa (30 mẫu, nửa còn lại ở notebook kia; cùng bộ chọn tất định). Mọi phép xoay đáp án. Bảy cấu hình: R5_C0 (native cùng phiên), C1..C5 (các ứng viên), C6 (đối chứng ngẫu nhiên cùng norm, 3 seed). Mỗi phép xoay mất khoảng 13 lượt forward, nên chạy lâu.

### Đọc kết quả thế nào (tiêu chí định trước)
Sau khi cả hai notebook xong: `python server_jobs/semantic_manage.py summary --files <zip1>/sem_chrono_s1.jsonl <zip2>/sem_chrono_s2.jsonl`. Tiêu chí viết trước (results/semantic_chrono/protocol.md): độ chính xác cân bằng cao hơn native >= 5 điểm; McNemar ghép cặp Holm p < 0,05; hơn đối chứng C6 >= 5 điểm; không task nào giảm quá 3 điểm; không dồn quá 75% dự đoán vào một chữ. Chưa đạt đủ thì không kết luận; đạt thì xác nhận trên bộ mới trước khi nói tới paper.

Không có nhãn nào được đưa vào mô hình; nhãn chỉ dùng để chấm điểm.

In [ ]:
# CELL 0 — kiểm tra Internet
import socket
for host in ('pypi.org', 'huggingface.co'):
    try:
        socket.create_connection((host, 443), timeout=8).close(); print('OK  ', host)
    except OSError as e:
        raise RuntimeError('Không kết nối được ' + host + '. Bật Internet: Session options -> Internet = On.') from e


In [ ]:
# CELL 1 — cài môi trường. Chạy một lần cho mỗi session Kaggle.
import os, sys, json, subprocess, importlib.metadata, venv, time, html, hashlib, zipfile, shutil
from pathlib import Path
from IPython.display import display, Video, Markdown, HTML, FileLink
assert Path('/kaggle').exists(), 'Dùng Kaggle Edit với GPU T4 x2 và Internet.'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['PYTHONIOENCODING']='utf-8'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
os.environ['FORCE_QWENVL_VIDEO_READER']='decord'  # torchvision.io.read_video was removed in Kaggle's build.
ROOT=Path('/kaggle/working/svhalluc_prompt_lab_runner');ROOT.mkdir(parents=True,exist_ok=True)
if 'OUT' not in globals():
    OUT=Path('/kaggle/working/svhalluc_prompt_lab_'+time.strftime('%Y%m%d_%H%M%S'))
    OUT.mkdir(parents=True,exist_ok=False)
SOURCES={'lab_client.py': '"""Standard-library client; the notebook kernel never imports/reloads the model."""\nimport json\nimport queue\nimport subprocess\nimport threading\nimport time\nfrom pathlib import Path\nfrom uuid import uuid4\n\nPREFIX=\'__SVHLAB_JSON__\'\n\n\nclass PromptLab:\n    def __init__(self, python, root, output, mock=False):\n        self.root=Path(root);self.output=Path(output);self.output.mkdir(parents=True,exist_ok=True)\n        self.log_path=self.output/\'worker.log\'\n        self.log=self.log_path.open(\'a\',encoding=\'utf-8\')\n        self.responses=queue.Queue()\n        command=[str(python),\'-u\',str(self.root/\'lab_worker.py\'),\'--output\',str(self.output)]\n        if mock:command.append(\'--mock\')\n        self.process=subprocess.Popen(command,cwd=str(self.root),stdin=subprocess.PIPE,stdout=subprocess.PIPE,\n            stderr=self.log,text=True,encoding=\'utf-8\',errors=\'replace\',bufsize=1)\n        self.reader=threading.Thread(target=self._read,daemon=True);self.reader.start()\n        self.ready=self._wait(None,3600)\n        if self.ready.get(\'event\')!=\'ready\':raise RuntimeError(self.ready)\n\n    def _read(self):\n        for line in self.process.stdout:\n            if line.startswith(PREFIX):\n                try:self.responses.put(json.loads(line[len(PREFIX):]))\n                except Exception:self.responses.put({\'event\':\'fatal\',\'error\':\'Malformed worker response\'})\n\n    def _wait(self, request_id, timeout):\n        started=time.monotonic();last_print=started\n        while time.monotonic()-started<timeout:\n            try:response=self.responses.get(timeout=1)\n            except queue.Empty:\n                if self.process.poll() is not None:raise RuntimeError(\'Worker stopped; see \'+str(self.log_path))\n                if time.monotonic()-last_print>=30:\n                    print(\'Đang tải/chạy model; log:\',self.log_path,flush=True);last_print=time.monotonic()\n                continue\n            if response.get(\'event\')==\'fatal\':raise RuntimeError(response.get(\'error\'))\n            if request_id is None or response.get(\'request_id\')==request_id:return response\n            # Late reply after a notebook interrupt is not reused for a newer prompt.\n        raise TimeoutError(\'Worker timed out; see \'+str(self.log_path))\n\n    def is_alive(self):return self.process.poll() is None\n\n    def request(self, payload, timeout=1800):\n        if not self.is_alive():raise RuntimeError(\'Run the model-loading cell again\')\n        request_id=uuid4().hex\n        self.process.stdin.write(json.dumps({**payload,\'request_id\':request_id},ensure_ascii=False)+\'\\n\')\n        self.process.stdin.flush()\n        response=self._wait(request_id,timeout)\n        if not response.get(\'ok\'):raise RuntimeError(response.get(\'error\'))\n        return response[\'result\']\n\n    def close(self):\n        if self.is_alive():self.process.terminate();self.process.wait(timeout=30)\n        self.log.close()\n', 'lab_native_video.py': '"""Native video-file input through Qwen\'s official utilities; no evidence extraction."""\nfrom pathlib import Path\n\nSYSTEM=\'You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech.\'\n# T4 SDPA cannot hold the default ~18k-token attention matrix. Keep Qwen\'s\n# temporal sampling and original MP4 path, while bounding only pixels per frame.\nVIDEO_MAX_PIXELS=160*28*28\n\n\ndef conversation_for(path, text, media_mode=\'av\', max_pixels=VIDEO_MAX_PIXELS):\n    if media_mode not in (\'av\',\'vision\',\'audio\'):raise ValueError(\'Unknown media mode\')\n    media=({\'type\':\'audio\',\'audio\':str(path)} if media_mode==\'audio\' else\n           {\'type\':\'video\',\'video\':str(path),\'max_pixels\':max_pixels})\n    return [\n        {\'role\':\'system\',\'content\':[{\'type\':\'text\',\'text\':SYSTEM}]},\n        {\'role\':\'user\',\'content\':[media,\n                                  {\'type\':\'text\',\'text\':text}]},\n    ]\n\n\ndef prepare_video(path, max_pixels=VIDEO_MAX_PIXELS):\n    from qwen_omni_utils import process_mm_info\n    path=Path(path).resolve()\n    if not path.is_file():raise FileNotFoundError(path)\n    # MP4 path, not caller-extracted frames. Keep Qwen\'s default temporal sampling;\n    # max_pixels per frame only bounds resolution for the T4 memory budget.\n    # The notebook selects Qwen\'s supported Decord reader because Kaggle\'s\n    # torchvision build no longer provides torchvision.io.read_video.\n    audios,images,videos,video_kwargs=process_mm_info(\n        conversation_for(path,\'\',max_pixels=max_pixels),use_audio_in_video=True,return_video_kwargs=True)\n    if images is not None:raise RuntimeError(\'Unexpected supplemental images\')\n    if audios is None or len(audios)!=1 or videos is None or len(videos)!=1:\n        raise RuntimeError(\'Expected one original video and its audio\')\n    # Transformers 4.57.6 Omni expects a scalar fps for this single video.\n    fps=video_kwargs[\'fps\']\n    if isinstance(fps,(list,tuple)):\n        if len(fps)!=1:raise RuntimeError(\'Expected one video fps\')\n        fps=fps[0]\n    return {\'path\':str(path),\'audio\':audios,\'videos\':videos,\'fps\':float(fps),\n            \'max_pixels_per_frame\':max_pixels}\n\n\ndef qwen_inputs(processor, prompt, media, media_mode=\'av\'):\n    if media_mode not in (\'av\',\'vision\',\'audio\'):raise ValueError(\'Unknown media mode\')\n    use_audio=media_mode in (\'av\',\'audio\')\n    use_video=media_mode in (\'av\',\'vision\')\n    rendered=processor.apply_chat_template(conversation_for(media[\'path\'],prompt,media_mode,media[\'max_pixels_per_frame\']),\n        add_generation_prompt=True,tokenize=False)\n    inputs=processor(text=rendered,audio=media[\'audio\'] if use_audio else None,images=None,videos=media[\'videos\'] if use_video else None,\n        return_tensors=\'pt\',padding=True,\n        videos_kwargs={\'fps\':media[\'fps\'],\'use_audio_in_video\':media_mode==\'av\'})\n    if (\'input_features\' in inputs)!=use_audio or (\'pixel_values_videos\' in inputs)!=use_video:\n        raise RuntimeError(\'Native inputs do not match the requested modalities\')\n    if \'pixel_values\' in inputs:raise RuntimeError(\'Unexpected supplemental image tokens\')\n    # HF Transformers takes this switch from videos_kwargs, not its top-level\n    # kwargs. Check that the resulting token sequence actually contains audio.\n    audio_token_id=processor.tokenizer.convert_tokens_to_ids(processor.audio_token)\n    audio_tokens=int((inputs[\'input_ids\']==audio_token_id).sum().item())\n    expected_audio_tokens=0\n    if use_audio:\n        if \'feature_attention_mask\' not in inputs:\n            raise RuntimeError(\'Audio feature mask missing; cannot verify audio-token alignment\')\n        input_lengths=(inputs[\'feature_attention_mask\'].sum(-1)-1)//2+1\n        expected_audio_tokens=int((((input_lengths-2)//2+1).clamp_min(0)).sum().item())\n    if audio_tokens!=expected_audio_tokens or (use_audio and audio_tokens==0):\n        raise RuntimeError(f\'Audio tokens missing/misaligned: found {audio_tokens}, expected {expected_audio_tokens}\')\n    expected_video_tokens=0\n    if use_video:\n        grid=inputs[\'video_grid_thw\']\n        merge=processor.video_processor.merge_size**2\n        expected_video_tokens=int((grid.prod(-1).sum()//merge).item())\n    video_token_id=processor.tokenizer.convert_tokens_to_ids(processor.video_token)\n    video_tokens=int((inputs[\'input_ids\']==video_token_id).sum().item())\n    if video_tokens!=expected_video_tokens or (use_video and video_tokens==0):\n        raise RuntimeError(f\'Video tokens missing/misaligned: found {video_tokens}, expected {expected_video_tokens}\')\n    return inputs,rendered\n\n\ndef move_inputs(inputs, model):\n    import torch\n    inputs=inputs.to(model.get_input_embeddings().weight.device)\n    for key in (\'input_features\',\'pixel_values_videos\'):\n        if key in inputs:inputs[key]=inputs[key].to(torch.float16)\n    return inputs\n', 'lab_setup.py': '"""Create/repair a venv without ensurepip; use the host pip to manage it."""\nimport importlib.metadata\nimport json\nimport subprocess\nimport sys\nimport venv\nfrom pathlib import Path\n\n\ndef prepare_environment(directory):\n    directory=Path(directory).resolve()\n    version=importlib.metadata.version(\'pip\')\n    parts=version.split(\'.\')\n    if tuple(int(part) for part in parts[:2]) < (22,3):\n        raise RuntimeError(\'Host pip >=22.3 is required for --python; found \'+version)\n    # Repeat even when the interpreter exists: an earlier ensurepip failure leaves\n    # an incomplete venv. No deletion, pip bootstrap or system package changes.\n    venv.EnvBuilder(system_site_packages=True,with_pip=False).create(directory)\n    python=directory/(\'Scripts/python.exe\' if sys.platform==\'win32\' else \'bin/python\')\n    completed=subprocess.run([str(python),\'-c\',\n        \'import json,sys; print(json.dumps({"prefix":sys.prefix,"base":sys.base_prefix}))\'],\n        check=True,capture_output=True,text=True,encoding=\'utf-8\')\n    info=json.loads(completed.stdout)\n    if Path(info[\'prefix\']).resolve()!=directory or info[\'prefix\']==info[\'base\']:\n        raise RuntimeError(\'The target interpreter is not the requested virtual environment\')\n    # Official pip --python also supports targets without their own pip module.\n    return python,[sys.executable,\'-m\',\'pip\',\'--python\',str(python)]\n', 'lab_worker.py': '"""Persistent, frozen Qwen Omni worker: one model load, fresh context for each prompt."""\nfrom __future__ import annotations\nimport argparse\nimport gc\nimport json\nimport re\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\nfrom uuid import uuid4\n\nWIRE = sys.stdout\nPREFIX = "__SVHLAB_JSON__"\nMODEL = "Qwen/Qwen2.5-Omni-3B"\nREVISION = "f75b40e3da2003cdd6e1829b1f420ca70797c34e"\n\n\ndef emit(value):\n    WIRE.write(PREFIX + json.dumps(value, ensure_ascii=False) + "\\n"); WIRE.flush()\n\n\ndef parse_final(text, complete):\n    """Read the explicit final choice, with support for older prompt formats."""\n    if not complete: return None\n    final_labels = re.findall(r"(?i)\\bFinal answer\\s*:", text)\n    if len(final_labels)>1:return None\n    final_match = re.search(r"(?i)\\bFinal answer\\s*:\\s*([AB](?:\\s*\\((?:Yes|No)\\))?)\\s*[.!?]?\\s*$",text)\n    if final_labels and not final_match:return None\n    final_fields=[final_match.group(1)] if final_match else []\n    fields = re.findall(r"(?im)^\\s*4\\.\\s*Original question\\s*:\\s*([^\\n]+)", text)\n    if len(fields)>1:return None\n    terminal = re.search(r"(?i)\\b(?:the\\s+)?answer\\s+is\\s+([AB](?:\\s*\\((?:Yes|No)\\))?)[.!]?\\s*$",text)\n    value=(final_fields[0] if final_fields else fields[0] if fields else terminal.group(1) if terminal else text).strip()\n    match=re.match(r"^(?:Answer\\s*:\\s*)?([AB])(?=$|[\\s().:—-])",value,re.I)\n    if not match or re.search(r"\\bor\\s+[AB](?:\\b|\\s*\\()",value,re.I):return None\n    letter=match.group(1).upper()\n    meaning=re.match(r"\\s*\\((Yes|No)\\)",value[match.end():],re.I)\n    if meaning and (letter,meaning.group(1).lower()) not in (("A","yes"),("B","no")):return None\n    return letter\n\n\ndef parse_baseline(text,complete):\n    if not complete:return None\n    value=text.strip()\n    match=re.match(r"^(?:(?:The\\s+)?(?:answer|option)\\s*(?:is|:)\\s*)?([AB])\\b",value,re.I)\n    if match:\n        if re.search(r"\\bor\\s+[AB]\\b",value,re.I):return None\n        letter=match.group(1).upper()\n        meaning=re.match(r"\\s*\\((Yes|No)\\)",value[match.end():],re.I)\n        if meaning and (letter,meaning.group(1).lower()) not in ((\'A\',\'yes\'),(\'B\',\'no\')):return None\n        return letter\n    match=re.match(r"^(Yes|No)\\b",value,re.I)\n    return (\'A\' if match.group(1).lower()==\'yes\' else \'B\') if match else None\n\n\ndef final_format_issues(text, complete):\n    issues = []\n    if not complete: issues.append("Output hit the token limit: final answer is not counted.")\n    if complete and parse_final(text, complete) is None:\n        issues.append("Could not identify one unambiguous Final answer: A (Yes) or B (No), or a supported older answer format.")\n    return issues\n\n\nclass Engine:\n    def __init__(self, output):\n        import torch\n        from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration\n        from transformers.models.qwen2_5_omni.configuration_qwen2_5_omni import Qwen2_5OmniThinkerConfig\n        self.output = Path(output); self.output.mkdir(parents=True, exist_ok=True)\n        torch.set_num_threads(2); torch.manual_seed(20261009)\n        if torch.cuda.device_count() < 2: raise RuntimeError("Enable Kaggle T4 x2")\n        config = Qwen2_5OmniThinkerConfig.from_pretrained(MODEL, revision=REVISION)\n        self.model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(\n            MODEL, revision=REVISION, config=config, torch_dtype=torch.float16,\n            attn_implementation="sdpa", device_map="balanced",\n            max_memory={0:"10GiB",1:"10GiB"}, low_cpu_mem_usage=True).eval()\n        for p in self.model.parameters(): p.requires_grad_(False)\n        if any(str(d) in ("cpu","disk") for d in self.model.hf_device_map.values()):\n            raise RuntimeError("Unexpected CPU offload; check both GPUs are available")\n        self.processor = Qwen2_5OmniProcessor.from_pretrained(MODEL, revision=REVISION)\n        self.meta = {"model":MODEL,"revision":REVISION,"device_map":self.model.hf_device_map,\n                     "runtime_parameters":sum(p.numel() for p in self.model.parameters()),\n                     "diagnostic_support":1}\n        self.media = self.row = None\n        (self.output / "model.json").write_text(json.dumps(self.meta, indent=2), encoding="utf-8")\n\n    def sample(self, request):\n        from lab_native_video import prepare_video\n        allowed = {"cmd","request_id","item_id","question","path"}\n        if set(request)-allowed: raise ValueError("Sample request has unexpected fields, possibly a label")\n        self.media = None; self.row = None\n        media = prepare_video(request[\'path\'])\n        self.media = media\n        self.row = {k:request[k] for k in ("item_id","question","path")}\n        result = {**self.row,\'frame_count\':int(media[\'videos\'][0].shape[0]),\n                  \'video_shape\':list(media[\'videos\'][0].shape),\'sampled_fps\':media[\'fps\'],\n                  \'audio_duration_s\':len(media[\'audio\'][0])/16000,\n                  \'preprocessing\':\'Official qwen-omni-utils 0.0.8, direct MP4 path, default temporal sampling, bounded max_pixels per frame for T4\',\n                  \'max_pixels_per_frame\':media[\'max_pixels_per_frame\'],\n                  \'same_media_for_both_conditions\':True}\n        (self.output/(request["item_id"]+"_media.json")).write_text(json.dumps(result,indent=2),encoding="utf-8")\n        return result\n\n    def generate(self, prompt, limit, baseline=False, media_mode=\'av\', diagnostic=False):\n        import torch\n        from lab_native_video import qwen_inputs, move_inputs\n        if self.media is None: raise RuntimeError("Select/load one sample before inference")\n        if not isinstance(limit,int) or not 16 <= limit <= 1024: raise ValueError("Token limit must be 16..1024")\n        inputs = None\n        started = time.perf_counter()\n        try:\n            inputs, rendered = qwen_inputs(self.processor, prompt, self.media,media_mode)\n            capacity=self.model.config.text_config.max_position_embeddings\n            if inputs[\'input_ids\'].shape[-1]+limit > capacity:\n                raise RuntimeError(f\'Native video + prompt exceed the model context ({capacity}); no media was silently truncated\')\n            inputs = move_inputs(inputs,self.model)\n            for i in range(torch.cuda.device_count()): torch.cuda.reset_peak_memory_stats(i)\n            eos = {self.processor.tokenizer.eos_token_id}\n            original_eos = self.model.generation_config.eos_token_id\n            if isinstance(original_eos,int): eos.add(original_eos)\n            elif original_eos: eos.update(original_eos)\n            eos.discard(None)\n            # No previous chat or past_key_values is supplied; only the model/media are reused.\n            with torch.inference_mode():\n                output = self.model.generate(**inputs, use_audio_in_video=media_mode==\'av\', do_sample=False,\n                    num_beams=1, use_cache=True, max_new_tokens=limit, eos_token_id=sorted(eos),\n                    pad_token_id=self.processor.tokenizer.pad_token_id,\n                    return_dict_in_generate=True, output_logits=False)\n            ids = output.sequences[0,inputs[\'input_ids\'].shape[-1]:]\n            complete = bool(len(ids) and int(ids[-1]) in eos)\n            raw = self.processor.tokenizer.decode(ids,skip_special_tokens=True).strip()\n            result = {"item_id":self.row[\'item_id\'],"question":self.row[\'question\'],"prompt":prompt,\n                "rendered_prompt":rendered,"raw":raw,"complete":complete,"generated_tokens":len(ids),\n                "max_new_tokens":limit,"input_tokens":int(inputs[\'input_ids\'].shape[-1]),\n                "prediction":None if diagnostic else parse_baseline(raw,complete) if baseline else parse_final(raw,complete),\n                "format_issues":final_format_issues(raw,complete) if not (baseline or diagnostic) else [],\n                "latency_s":time.perf_counter()-started,\n                "peak_allocated_GiB":[round(torch.cuda.max_memory_allocated(i)/2**30,3) for i in range(torch.cuda.device_count())],\n                "method":"diagnostic" if diagnostic else "baseline" if baseline else "prompt","fresh_context":True,\n                "media_mode":media_mode,"audio_provided":media_mode in (\'av\',\'audio\'),\n                "video_provided":media_mode in (\'av\',\'vision\'),\n                "max_pixels_per_frame":self.media[\'max_pixels_per_frame\'],\n                "video_shape":list(self.media[\'videos\'][0].shape) if media_mode!=\'audio\' else None,\n                "run_id":uuid4().hex[:12]}\n            result[\'decision_interface\']=\'Read the freely generated answer; no logit adjustment or forced A/B selection\'\n            result[\'media_preprocessing\']=\'official_direct_video_defaults\'\n            path=self.output/(self.row[\'item_id\']+\'_\'+result[\'method\']+\'_\'+result[\'run_id\']+\'.json\')\n            path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n            result[\'result_path\']=str(path)\n            return result\n        finally:\n            inputs=None; gc.collect(); torch.cuda.empty_cache()\n\n    def dispatch(self, request):\n        command=request[\'cmd\']\n        if command==\'sample\':return self.sample(request)\n        if command==\'baseline\':\n            return self.generate(self.row[\'question\'],request.get(\'max_new_tokens\',384),True)\n        if command==\'prompt\':return self.generate(request[\'prompt\'],request.get(\'max_new_tokens\',384))\n        if command==\'diagnostic\':\n            if set(request)-{\'cmd\',\'request_id\',\'prompt\',\'media_mode\',\'max_new_tokens\',\'max_pixels\'}:\n                raise ValueError(\'Unexpected diagnostic fields; no labels are accepted\')\n            mode=request.get(\'media_mode\',\'av\')\n            if mode not in (\'av\',\'vision\',\'audio\'):raise ValueError(\'Unknown media mode\')\n            if self.media is None:raise RuntimeError(\'Select one sample first\')\n            current=self.media\n            pixels=request.get(\'max_pixels\',current[\'max_pixels_per_frame\'])\n            if pixels not in (current[\'max_pixels_per_frame\'],2*current[\'max_pixels_per_frame\']):\n                raise ValueError(\'Diagnostic resolution must be unchanged or doubled\')\n            try:\n                if pixels!=current[\'max_pixels_per_frame\']:\n                    from lab_native_video import prepare_video\n                    self.media=prepare_video(self.row[\'path\'],pixels)\n                    if self.media[\'fps\']!=current[\'fps\'] or self.media[\'videos\'][0].shape[0]!=current[\'videos\'][0].shape[0]:\n                        raise RuntimeError(\'Resolution probe changed temporal sampling\')\n                return self.generate(request[\'prompt\'],request.get(\'max_new_tokens\',192),media_mode=mode,diagnostic=True)\n            finally:self.media=current\n        if command==\'ping\':return self.meta\n        raise ValueError(\'Unknown command: \'+command)\n\n\ndef main():\n    sys.stdout = sys.stderr  # Library prints/progress cannot corrupt the JSON protocol.\n    parser=argparse.ArgumentParser();parser.add_argument(\'--output\',required=True);parser.add_argument(\'--mock\',action=\'store_true\')\n    args=parser.parse_args()\n    try:\n        engine=Engine(args.output) if not args.mock else None\n        emit({\'event\':\'ready\',\'model\':MODEL,\'revision\':REVISION,\'mock\':args.mock})\n    except Exception:\n        emit({\'event\':\'fatal\',\'error\':traceback.format_exc()});return\n    for line in sys.stdin:\n        request_id=None\n        try:\n            request=json.loads(line);request_id=request.get(\'request_id\')\n            if request[\'cmd\']==\'close\':emit({\'request_id\':request_id,\'ok\':True,\'result\':{}});break\n            result=engine.dispatch(request) if engine else {\'mock\':True,\'echo\':request}\n            emit({\'request_id\':request_id,\'ok\':True,\'result\':result})\n        except Exception:\n            emit({\'request_id\':request_id,\'ok\':False,\'error\':traceback.format_exc()})\n\n\nif __name__==\'__main__\':main()\n'}
for name,source in SOURCES.items():(ROOT/name).write_text(source,encoding='utf-8')
HISTORICAL={'item_id': 'svh_0630', 'question': 'Is the thinly sliced scallions visible in the video? (A) Yes (B) No', 'prediction': 'A', 'gold': 'B', 'prompt': 'Is the thinly sliced scallions visible in the video? (A) Yes (B) No\nAnswer with exactly one option letter, A or B. Do not write an explanation.', 'method': 'AV_baseline', 'correct': 'False', 'raw': 'A', 'condition': 'question_only_historical_AV_baseline', 'source_zip': 'results.zip', 'source_member': 'svhalluc_external_evidence_20261008_070149/predictions.csv', 'note': 'Historical question-only AV_baseline log: prediction A, gold B. The notebook freshly reruns the baseline and the edited prompt on this sample; outputs can vary with runtime/model preprocessing.'}
(OUT/'historical_default.json').write_text(json.dumps(HISTORICAL,ensure_ascii=False,indent=2),encoding='utf-8')
if 'LAB' in globals() and LAB.is_alive():
    raise RuntimeError('Model đang chạy. Chỉ chạy cell PROMPT; nếu cần cài lại, gọi LAB.close() trước.')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from lab_setup import prepare_environment
ENV=ROOT/'env'
PYTHON,PIP_COMMAND=prepare_environment(ENV)
print('Venv sẵn sàng (không dùng ensurepip):',PYTHON,flush=True)
constraints=[]
for package in ('torch','torchvision','torchaudio'):
    try:constraints.append(package+'=='+importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:pass
(ROOT/'keep_torch.txt').write_text('\n'.join(constraints)+'\n',encoding='utf-8')
packages=['transformers==4.57.6','accelerate==1.12.0','huggingface-hub==0.36.0','qwen-omni-utils==0.0.8','decord==0.6.0',
          'av==16.0.1','librosa==0.11.0','soundfile>=0.12,<0.14','numpy>=2,<2.4',
          'Pillow>=10,<13','sentencepiece>=0.2,<0.3','protobuf>=5,<7']
command=[*PIP_COMMAND,'install','--only-binary=:all:',
         '--constraint',str(ROOT/'keep_torch.txt'),*packages]
with (OUT/'install.log').open('a',encoding='utf-8') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,encoding='utf-8',errors='replace')
    for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
    code=process.wait()
if code:raise RuntimeError('Package installation failed; see '+str(OUT/'install.log'))
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get','-qq','update'],check=True)
    subprocess.run(['apt-get','-qq','install','-y','ffmpeg'],check=True)
subprocess.run([str(PYTHON),'-c','import torch, av, decord; from qwen_omni_utils.v2_5.vision_process import get_video_reader_backend; from transformers import Qwen2_5OmniThinkerForConditionalGeneration; assert torch.cuda.device_count()>=1; assert get_video_reader_backend()=="decord"; print("torch",torch.__version__,"decord",decord.__version__)'],check=True)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from lab_client import PromptLab
print('Setup ready. Kết quả và log:',OUT)


In [ ]:
# CELL 2 — ghi mã nguồn
JOB = 'sem_chrono.py'
ZIPNAME = 'semchrono_s1'
PARAMS = {'SEM_SHARD': 0, 'SEM_PER_CELL': 10, 'SEM_SEED': 31337, 'HOURS': 10.5}
SRC = {'speech_gate.py': '"""Speech-anchored attention gating for Qwen2.5-Omni Thinker (single forward pass, no training, no re-encoding).\n\nQwen2.5-Omni gives every audio and video token a temporal rope id (25 ids per second, shared clock).\nGiven a time window [s0, s1] (seconds), we find the video tokens whose temporal patch overlaps the window and,\nfor every text query position AFTER the media block (question + assistant prefix), we add a bias to the attention\nlogits of video keys OUTSIDE the window:  hard = -inf (attention-level crop), soft = -beta.\nMedia tokens themselves attend as usual, so the visual and audio representations are unchanged; only what the\nquestion/answer positions read from is steered.\n"""\nimport torch\n\n\ndef media_layout(model, proc, inputs):\n    """Return position_ids (3,1,L), per-token time in seconds for video tokens, and index sets."""\n    ids = inputs[\'input_ids\']\n    am = inputs[\'attention_mask\']\n    afl = inputs[\'feature_attention_mask\'].sum(-1) if \'feature_attention_mask\' in inputs else None\n    spg = inputs.get(\'video_second_per_grid\')\n    pos, _ = model.get_rope_index(ids, None, inputs.get(\'video_grid_thw\'), am, True, afl, spg)\n    tok = proc.tokenizer\n    vid_id = tok.convert_tokens_to_ids(proc.video_token)\n    aud_id = tok.convert_tokens_to_ids(proc.audio_token)\n    is_v = (ids[0] == vid_id)\n    is_a = (ids[0] == aud_id)\n    t = pos[0, 0].float()\n    base = t[is_a].min() if is_a.any() else t[is_v].min()\n    sec = (t - base) / 25.0                       # temporal rope id -> seconds on the shared clock\n    last_media = int(torch.nonzero(is_v | is_a).max())\n    return pos, sec, is_v, is_a, last_media\n\n\ndef gated_mask(L, is_v, sec, last_media, window, mode, beta, dtype, device, spg=None, scope=\'all\', is_a=None, block_audio=False):\n    """scope=\'text\': only positions after the media block are gated (leaks through media tokens that already read the\n    out-of-window frames). scope=\'all\': no position except the out-of-window video tokens themselves may read them,\n    i.e. an attention-level crop that keeps positions, audio and the full token sequence."""\n    neg = torch.finfo(dtype).min\n    m = torch.zeros(L, L, dtype=torch.float32, device=device)\n    m = m.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1), float(\'-inf\'))\n    if mode != \'none\':\n        s0, s1 = window\n        width = float(spg) if spg is not None else 0.0   # a temporal patch covers [t, t + second_per_grid)\n        inside = (sec + width > s0) & (sec < s1)\n        out_v = is_v.to(device) & ~inside.to(device)\n        rows = (torch.arange(L, device=device) > last_media) if scope == \'text\' else ~out_v\n        bias = float(\'-inf\') if mode == \'hard\' else -float(beta)\n        m[rows.unsqueeze(1) & out_v.unsqueeze(0)] += bias\n    if block_audio:\n        # audio-blind decision: no non-audio position may read audio tokens (audio tokens still read each other)\n        aud = is_a.to(device)\n        m[(~aud).unsqueeze(1) & aud.unsqueeze(0)] = float(\'-inf\')\n    m = m.clamp(min=neg).to(dtype)\n    return m.view(1, 1, L, L)\n\n\ndef binding_mask(L, is_v, is_a, sec, dtype, device, chunk_s=2.0, delta=0, restrict=True, bidir=False, base=None):\n    """Synchronous audio-visual binding inside the media block (text rows are never changed).\n    Qwen2.5-Omni interleaves every 2-s chunk as [video_c, audio_c] under a causal mask, so\n      - video_c can never read audio_c (the sound of the SAME moment comes later in the sequence), and\n      - audio_c (and video_c) can read the other modality of ALL earlier moments.\n    restrict=True : a cross-modal edge (audio<->video) is kept only if the two chunks differ by <= delta\n    bidir=True    : additionally let video_c read audio_c\' for |c - c\'| <= delta even though it comes later (prefill only)\n    Within-modality attention and everything involving text tokens keep the original causal pattern."""\n    m = torch.zeros(L, L, dtype=torch.float32, device=device) if base is None else base.clone().float()\n    if base is None:\n        m = m.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1), float(\'-inf\'))\n    v, a = is_v.to(device), is_a.to(device)\n    ch = torch.floor(sec.to(device) / chunk_s)\n    near = (ch.unsqueeze(1) - ch.unsqueeze(0)).abs() <= delta\n    cross = (v.unsqueeze(1) & a.unsqueeze(0)) | (a.unsqueeze(1) & v.unsqueeze(0))\n    if restrict:\n        m[cross & ~near] = float(\'-inf\')\n    if bidir:\n        later = torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1)\n        m[(v.unsqueeze(1) & a.unsqueeze(0)) & near & later] = 0.0\n    neg = torch.finfo(dtype).min\n    return m.clamp(min=neg).to(dtype).view(1, 1, L, L)\n\n\ndef balance_mask(L, is_v, is_a, last_media, dtype, device, strength=1.0):\n    """Modality-balanced readout: question/answer rows get an additive bias on audio keys equal to\n    strength * log(n_video / n_audio), so that with equal content scores the two streams receive equal total attention\n    (Qwen2.5-Omni gives ~144 video tokens vs ~25 audio tokens per second)."""\n    m = torch.zeros(L, L, dtype=torch.float32, device=device)\n    m = m.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1), float(\'-inf\'))\n    v, a = is_v.to(device), is_a.to(device)\n    b = strength * float(torch.log(v.sum().float() / a.sum().float().clamp(min=1)))\n    rows = torch.arange(L, device=device) > last_media\n    m[rows.unsqueeze(1) & a.unsqueeze(0)] += b\n    return m.clamp(min=torch.finfo(dtype).min).to(dtype).view(1, 1, L, L)\n\n\ndef bound_last_logits(model, proc, inputs, delta=0, restrict=True, bidir=False):\n    pos, sec, is_v, is_a, last_media = media_layout(model, proc, inputs)\n    L = inputs[\'input_ids\'].shape[1]\n    emb = model.get_input_embeddings().weight\n    mask = binding_mask(L, is_v, is_a, sec, emb.dtype, emb.device, delta=delta, restrict=restrict, bidir=bidir)\n    kw = {k: v for k, v in inputs.items() if k != \'attention_mask\'}\n    with torch.no_grad():\n        out = model(**kw, attention_mask=mask, position_ids=pos.to(emb.device), use_audio_in_video=True)\n    return out.logits[0, -1].float()\n\n\ndef gated_last_logits(model, proc, inputs, window=None, mode=\'none\', beta=0.0, use_audio_in_video=True, scope=\'all\', block_audio=False):\n    """Forward pass with explicit 3D rope ids and a 4D additive mask; returns last-position logits (float32)."""\n    pos, sec, is_v, is_a, last_media = media_layout(model, proc, inputs)\n    L = inputs[\'input_ids\'].shape[1]\n    emb = model.get_input_embeddings().weight\n    spg = inputs.get(\'video_second_per_grid\')\n    spg = float(spg.reshape(-1)[0]) if spg is not None else None\n    mask = gated_mask(L, is_v, sec, last_media, window, mode, beta, emb.dtype, emb.device, spg, scope, is_a, block_audio)\n    kw = {k: v for k, v in inputs.items() if k != \'attention_mask\'}\n    with torch.no_grad():\n        out = model(**kw, attention_mask=mask, position_ids=pos.to(emb.device), use_audio_in_video=use_audio_in_video)\n    return out.logits[0, -1].float(), dict(n_video=int(is_v.sum()), n_audio=int(is_a.sum()), last_media=last_media,\n                                           video_sec_range=[float(sec[is_v].min()), float(sec[is_v].max())])\n', 'kaggle_rootcause.py': '"""Root-cause completion for SVHalluc TEMPORAL (Qwen2.5-Omni-3B). Usage:\n  python kaggle_rootcause.py ROOT OUT PER_LABEL SHARD_ID NUM_SHARDS\nPer item (TA/TF/CMTB), all scored with rotated option orders and summed per option content:\n  base                  native model                                                        -> reference answer\n  para1, para2          two paraphrases of the question                                    -> #12 question wording\n  prior                 black video + silent audio of the same length                     -> #13 content-free prior\n  vision_only           original video, audio removed                                     -> modality reliance\n  audio_only            black video + original audio                                      -> modality reliance\n  (CMTB) gate, audio_block, gate_audio, crop                                               -> #9/#10 per-error decomposition\n  hear (text) + whisper (analysis-only reference transcript of the speech window)          -> #4 speech content\n  hidden states (last token, mean of question rows) at several layers from the base run    -> #14 representation-action gap\n  (CMTB, base wrong and gate_audio right) activation patching sweep                        -> where the error enters\nShard 0 additionally runs:\n  synthetic text-only relation questions (no media)                                        -> #11 relation reasoning\n  eager attention of the question rows for the first 24 items                              -> #8 front-loading at scale\nLabels are used only for scoring. Whisper is an analysis tool only; it never feeds the model."""\nimport os, sys, json, re, subprocess, time, random\nfrom pathlib import Path\nimport numpy as np\nimport torch\n\nos.environ[\'FORCE_QWENVL_VIDEO_READER\'] = \'decord\'\nROOT, OUT = Path(sys.argv[1]), Path(sys.argv[2])\nPER_LABEL, SHARD, NSHARD = int(sys.argv[3]), int(sys.argv[4]), int(sys.argv[5])\nMAX_DUR = float(os.environ.get(\'LAB_MAX_DUR\', 42.0 if torch.cuda.device_count() >= 2 else 28.0))\nsys.path.insert(0, str(ROOT))\nimport av, librosa\nfrom huggingface_hub import hf_hub_download\n\nDATASET, REVISION = \'zcs15/SVHalluc\', \'211b4f14ab66ae9557b69e302d1f090e0e610839\'\nMODEL, MREV = \'Qwen/Qwen2.5-Omni-3B\', \'f75b40e3da2003cdd6e1829b1f420ca70797c34e\'\nBS = chr(92); FF = os.environ.get(\'FFMPEG_BIN\', \'ffmpeg\')\nLAYERS_KEEP = [6, 12, 15, 18, 21, 24, 27, 30, 33, 36]\n\n\ndef decode_audio(path):\n    chunks = []\n    with av.open(str(path)) as c:\n        rs = av.AudioResampler(format=\'fltp\', layout=\'mono\', rate=16000)\n        for fr in c.decode(audio=0):\n            for p in rs.resample(fr): chunks.append(p.to_ndarray().reshape(-1))\n        for p in rs.resample(None): chunks.append(p.to_ndarray().reshape(-1))\n    return np.concatenate(chunks).astype(\'float32\', copy=False)\n\n\n_orig = librosa.load\ndef _load(path, sr=None, offset=0.0, duration=None, **k):\n    if str(path).lower().endswith(\'.mp4\'):\n        a = decode_audio(path)\n        return a[int(sr * offset): None if duration is None else int(sr * (offset + duration))], sr\n    return _orig(path, sr=sr, offset=offset, duration=duration, **k)\nlibrosa.load = _load\n\nfrom lab_native_video import prepare_video, qwen_inputs, move_inputs, SYSTEM\nfrom speech_gate import media_layout, gated_mask\nfrom transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor\n\nCLIPS = OUT / \'clips\'; CLIPS.mkdir(parents=True, exist_ok=True)\nHID = OUT / \'hidden\'; HID.mkdir(parents=True, exist_ok=True)\nENC = [\'-c:v\', \'libx264\', \'-preset\', \'ultrafast\', \'-crf\', \'24\', \'-pix_fmt\', \'yuv420p\', \'-c:a\', \'aac\']\n\n\ndef fetch(name):\n    local = os.environ.get(\'LAB_LOCAL_DATA\')\n    if local and (Path(local) / name).exists(): return Path(local) / name\n    return Path(hf_hub_download(DATASET, name, repo_type=\'dataset\', revision=REVISION))\n\n\ndef ffmpeg(*args):\n    r = subprocess.run([FF, \'-hide_banner\', \'-y\', *args], capture_output=True)\n    if r.returncode != 0 and b\'fps_mode\' in r.stderr:      # older ffmpeg builds only know -vsync\n        r = subprocess.run([FF, \'-hide_banner\', \'-y\', *[\'-vsync\' if a == \'-fps_mode\' else a for a in args]], capture_output=True)\n    if r.returncode != 0: raise RuntimeError(r.stderr.decode(errors=\'replace\')[-400:])\n\n\ndef cut(src, start, end, tag):\n    dst = CLIPS / f\'{Path(src).stem}_{tag}.mp4\'\n    if not dst.exists():\n        ffmpeg(\'-ss\', f\'{start:.2f}\', \'-to\', f\'{end:.2f}\', \'-i\', str(src), \'-map\', \'0:v:0\', \'-map\', \'0:a:0\', \'-vf\', \'fps=30\', \'-fps_mode\', \'cfr\', *ENC,\n               \'-af\', \'aresample=async=1:first_pts=0\', str(dst))\n    return dst\n\n\ndef video_size(src):\n    with av.open(str(src)) as c:\n        s = c.streams.video[0]; return s.width, s.height\n\n\ndef blank_video(src, dur, keep_audio):\n    """black frames of the same size/length; either the original audio or digital silence."""\n    w, h = video_size(src)\n    dst = CLIPS / f\'{Path(src).stem}_{"blackA" if keep_audio else "prior"}.mp4\'\n    if not dst.exists():\n        if keep_audio:\n            ffmpeg(\'-f\', \'lavfi\', \'-i\', f\'color=black:s={w}x{h}:r=30:d={dur:.2f}\', \'-i\', str(src), \'-map\', \'0:v:0\', \'-map\', \'1:a:0\', \'-shortest\', *ENC, str(dst))\n        else:\n            ffmpeg(\'-f\', \'lavfi\', \'-i\', f\'color=black:s={w}x{h}:r=30:d={dur:.2f}\', \'-f\', \'lavfi\', \'-i\', f\'anullsrc=r=16000:cl=mono\',\n                   \'-t\', f\'{dur:.2f}\', \'-map\', \'0:v:0\', \'-map\', \'1:a:0\', *ENC, str(dst))\n    return dst\n\n\ndef speech_window(a):\n    hop = 800; n = len(a) // hop\n    db = 20 * np.log10(np.sqrt((a[:n * hop].reshape(n, hop) ** 2).mean(1) + 1e-12))\n    lo, hi = np.percentile(db, 10), np.percentile(db, 95)\n    act = db > lo + 0.5 * (hi - lo)\n    if act.sum() == 0: return None\n    t = (np.arange(n) + 0.5) * 0.05; idx = np.where(act)[0]\n    return float(t[idx[0]]), float(t[idx[-1]])\n\n\ndef parse_question(q):\n    parts = re.split(r\'\\s*\\(([ABC])\\)\\s*\', q)\n    return parts[0].strip(), [parts[i + 1].strip() for i in range(1, len(parts) - 1, 2)]\n\n\ndef rotations(k): return [tuple((s + j) % k for j in range(k)) for s in range(k)]\ndef render(stem, texts, perm): return stem + \' \' + \' \'.join(f\'({L}) {texts[j]}\' for L, j in zip(\'ABC\', perm))\n\n\nPARA = {\'TA\': [\'Does the action that the speaker talks about happen in the video at the same moment the speaker is talking?\',\n               \'Look at the video only while the speech is playing. Is the narrated action visible during that time?\'],\n        \'TF\': [\'The speaker describes an action. Compared with the time the speaker is talking, does that action appear in the video earlier, at the same time, or later?\',\n               \'When does the narrated action happen in the video relative to the speech?\'],\n        \'CMTB\': [\'Which action is visible in the video while the person is speaking?\', \'During the speech, what is being done on screen?\']}\n\nMODEL_ = PROC = LETTERS = None\n\n\ndef load_model():\n    cache = os.environ.get(\'LAB_CACHE\')\n    kw = dict(revision=MREV, torch_dtype=torch.float16, attn_implementation=\'sdpa\', cache_dir=cache)\n    if os.environ.get(\'LAB_NF4\'):\n        from transformers import BitsAndBytesConfig\n        kw.update(device_map={\'\': 0}, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                  bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True))\n    elif torch.cuda.device_count() >= 2: kw.update(device_map=\'balanced\', max_memory={0: \'10GiB\', 1: \'10GiB\'})\n    else: kw.update(device_map={\'\': 0})\n    m = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(MODEL, **kw).eval()\n    return m, Qwen2_5OmniProcessor.from_pretrained(MODEL, revision=MREV, cache_dir=cache)\n\n\ndef masked_forward(inp, mask_kind, window, record_layers=None, patch=None):\n    """mask_kind: plain | gate | audio_block | gate_audio. record_layers -> dict(layer -> (last_token, mean_question_rows, question_rows))."""\n    emb = MODEL_.get_input_embeddings().weight\n    pos, sec, is_v, is_a, last_media = media_layout(MODEL_, PROC, inp)\n    L = inp[\'input_ids\'].shape[1]\n    spg = inp.get(\'video_second_per_grid\'); spg = float(spg.reshape(-1)[0]) if spg is not None else None\n    tg = mask_kind in (\'gate\', \'gate_audio\'); ab = mask_kind in (\'audio_block\', \'gate_audio\')\n    mask = gated_mask(L, is_v, sec, last_media, window, \'hard\' if tg else \'none\', 0.0, emb.dtype, emb.device, spg, \'all\', is_a, ab)\n    kw = {k: v for k, v in inp.items() if k != \'attention_mask\'}\n    rows_q = (torch.arange(L) > last_media)\n    store, hooks = {}, []\n    layers = MODEL_.model.layers\n    if record_layers or patch is not None:\n        for li in range(len(layers)):\n            def hook(mod, args, out, li=li):\n                h = out[0]\n                rq = rows_q.to(h.device)\n                if patch is not None and patch[0] == li:\n                    h = h.clone(); h[0, rq] = patch[1].to(h.device, h.dtype); out = (h,) + tuple(out[1:])\n                if record_layers and (li + 1) in record_layers:\n                    store[li + 1] = (h[0, -1].float().cpu(), h[0, rq].float().mean(0).cpu(), h[0, rq].detach().float().cpu())\n                return out\n            hooks.append(layers[li].register_forward_hook(hook))\n    try:\n        with torch.no_grad():\n            lg = MODEL_(**kw, attention_mask=mask, position_ids=pos.to(emb.device), use_audio_in_video=True).logits[0, -1].float()\n    finally:\n        for h in hooks: h.remove()\n    return lg, store\n\n\ndef scores(media, stem, texts, mode=\'av\', mask_kind=None, window=None):\n    k = len(texts); tot = np.zeros(k)\n    for perm in rotations(k):\n        inp, _ = qwen_inputs(PROC, render(stem, texts, perm), media, mode); inp = move_inputs(inp, MODEL_)\n        if mask_kind is None:\n            with torch.no_grad(): lg = MODEL_(**inp, use_audio_in_video=(mode == \'av\')).logits[0, -1].float()\n        else:\n            lg, _ = masked_forward(inp, mask_kind, window)\n        lp = torch.log_softmax(lg[LETTERS[:k]].cpu(), -1).numpy()\n        for pos, j in enumerate(perm): tot[j] += lp[pos]\n        del inp\n    torch.cuda.empty_cache()\n    return tot.round(3).tolist()\n\n\nHEAR_Q = \'What action does the speaker say is being done? Answer with a short phrase (verb and object) only.\'\nWHISPER = None\n\n\ndef hear(media):\n    inp, _ = qwen_inputs(PROC, HEAR_Q, media, \'audio\'); inp = move_inputs(inp, MODEL_)\n    with torch.no_grad(): o = MODEL_.generate(**inp, max_new_tokens=16, do_sample=False)\n    return PROC.batch_decode(o[:, inp[\'input_ids\'].shape[1]:], skip_special_tokens=True)[0].strip().split(\'\\n\')[0][:120]\n\n\ndef whisper_text(a, s0, s1):\n    global WHISPER\n    if os.environ.get(\'LAB_NO_WHISPER\'): return None\n    if WHISPER is None:\n        from transformers import pipeline\n        WHISPER = pipeline(\'automatic-speech-recognition\', model=\'openai/whisper-small\', device=0, torch_dtype=torch.float16)\n    seg = a[int(s0 * 16000): int(s1 * 16000)]\n    return WHISPER({\'raw\': seg, \'sampling_rate\': 16000}, generate_kwargs={\'language\': \'en\'})[\'text\'].strip()\n\n\ndef patch_sweep(media, stem, texts, window):\n    """Original option order only. Returns plain/clean letter logits and forward/reverse patching per layer."""\n    inp, _ = qwen_inputs(PROC, render(stem, texts, tuple(range(len(texts)))), media, \'av\'); inp = move_inputs(inp, MODEL_)\n    k = len(texts); allL = list(range(1, len(MODEL_.model.layers) + 1))\n    lg_p, Sp = masked_forward(inp, \'plain\', window, record_layers=allL)\n    lg_c, Sc = masked_forward(inp, \'gate_audio\', window, record_layers=allL)\n    let = lambda lg: [round(float(lg[t]), 3) for t in LETTERS[:k]]\n    res = dict(plain=let(lg_p), clean=let(lg_c), fwd={}, rev={})\n    for li in list(range(0, len(MODEL_.model.layers), 3)) + [len(MODEL_.model.layers) - 1]:\n        res[\'fwd\'][li] = let(masked_forward(inp, \'plain\', window, patch=(li, Sc[li + 1][2]))[0])\n        res[\'rev\'][li] = let(masked_forward(inp, \'gate_audio\', window, patch=(li, Sp[li + 1][2]))[0])\n    del inp; torch.cuda.empty_cache()\n    return res\n\n\ndef eager_attention(media, stem, texts, window, dur):\n    """per layer: attention mass (summed over question rows, mean over heads) to video first/second half, inside/outside window, audio."""\n    inp, _ = qwen_inputs(PROC, render(stem, texts, tuple(range(len(texts)))), media, \'av\'); inp = move_inputs(inp, MODEL_)\n    pos, sec, is_v, is_a, last_media = media_layout(MODEL_, PROC, inp)\n    L = inp[\'input_ids\'].shape[1]; rows_q = torch.arange(L) > last_media\n    sec_c, vi, ai = sec.cpu(), is_v.cpu(), is_a.cpu()\n    spg = float(inp[\'video_second_per_grid\'].reshape(-1)[0])\n    inside = (sec_c + spg > window[0]) & (sec_c < window[1])\n    groups = {\'v_first_half\': vi & (sec_c < dur / 2), \'v_second_half\': vi & (sec_c >= dur / 2), \'v_in\': vi & inside, \'v_out\': vi & ~inside, \'audio\': ai}\n    counts = {g: int(m.sum()) for g, m in groups.items()}\n    stats = {}\n    def mk(li):\n        def hook(mod, args, out):\n            w = out[1]\n            if w is None: return\n            dv = w.device; q = w[0][:, rows_q.to(dv)].float().sum(1).mean(0)\n            stats[li] = {g: round(float(q[m.to(dv)].sum()), 5) for g, m in groups.items()}\n            return (out[0], None)\n        return hook\n    MODEL_.set_attn_implementation(\'eager\')\n    hooks = [MODEL_.model.layers[li].self_attn.register_forward_hook(mk(li)) for li in range(len(MODEL_.model.layers))]\n    try:\n        with torch.no_grad(): MODEL_(**inp, use_audio_in_video=True)\n    finally:\n        for h in hooks: h.remove()\n        MODEL_.set_attn_implementation(\'sdpa\')\n    del inp; torch.cuda.empty_cache()\n    return dict(counts=counts, per_layer=[stats[li] for li in sorted(stats)])\n\n\ndef text_only_scores(stem, texts):\n    k = len(texts); tot = np.zeros(k)\n    for perm in rotations(k):\n        conv = [{\'role\': \'system\', \'content\': [{\'type\': \'text\', \'text\': SYSTEM}]}, {\'role\': \'user\', \'content\': [{\'type\': \'text\', \'text\': render(stem, texts, perm)}]}]\n        text = PROC.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)\n        inp = PROC(text=text, return_tensors=\'pt\').to(MODEL_.get_input_embeddings().weight.device)\n        with torch.no_grad(): lg = MODEL_(**inp).logits[0, -1].float()\n        lp = torch.log_softmax(lg[LETTERS[:k]].cpu(), -1).numpy()\n        for pos, j in enumerate(perm): tot[j] += lp[pos]\n    return tot.round(3).tolist()\n\n\nACTIONS = [\'pour the oil into the pan\', \'chop the onions\', \'add salt to the soup\', \'stir the sauce\', \'slice the tomatoes\',\n           \'crack the eggs into the bowl\', \'mash the potatoes\', \'spread butter on the bread\', \'fry the bacon\', \'mix the flour and water\']\n\n\nTENSES = {\'future\': \'I will {a}\', \'present\': \'Now I am going to {a} right now\', \'past\': \'I have already finished: {a}\', \'neutral\': \'{a}\'}\n\n\ndef synthetic_relation():\n    """relation (Past/Present/Future) x speech tense; text-only facts, no media. Tests reasoning and the tense shortcut."""\n    rng = random.Random(0); out = []\n    for tense, tmpl in TENSES.items():\n        for n in range(30):\n            act = rng.choice(ACTIONS); s0 = round(rng.uniform(10, 20), 1); sd = round(rng.uniform(2, 5), 1)\n            rel = n % 3                      # 0 Past, 1 Present, 2 Future\n            if rel == 1: e0 = round(s0 + rng.uniform(-0.5, 0.5), 1)\n            elif rel == 0: e0 = round(s0 - rng.uniform(6, 9), 1)\n            else: e0 = round(s0 + sd + rng.uniform(3, 9), 1)\n            ed = round(rng.uniform(2, 5), 1); said = tmpl.format(a=act)\n            facts = (f\'In a cooking video, a person says "{said}" from {s0} to {round(s0 + sd, 1)} seconds. \'\n                     f\'The action "{act}" is actually shown in the video from {e0} to {round(e0 + ed, 1)} seconds.\')\n            tf = text_only_scores(facts + \' Relative to the moment when the speech is heard, when does the narrated event occur in the video?\', [\'Past\', \'Present\', \'Future\'])\n            tf_plain = text_only_scores(facts + \' Is the action shown before, during, or after the speech?\', [\'Before\', \'During\', \'After\'])\n            ta = text_only_scores(facts + \' When the speech is heard, is the narrated event happening at the same time in the video?\', [\'Yes\', \'No\'])\n            out.append(dict(tense=tense, rel=rel, tf=tf, tf_plain=tf_plain, ta=ta, ta_gold=0 if rel == 1 else 1))\n    return out\n\n\ndef run_item(task, r, shard_attention):\n    orig = fetch(r[\'video_path\'].replace(BS, \'/\'))\n    full = cut(orig, 0, 10 ** 4, \'full\')\n    a = decode_audio(full); dur = len(a) / 16000\n    if dur > MAX_DUR: raise RuntimeError(\'skipped: longer than MAX_DUR\')\n    w = speech_window(a)\n    if w is None: raise RuntimeError(\'no speech activity\')\n    s0, s1 = w; win = (s0, s1)\n    media = prepare_video(full)\n    stem, texts = parse_question(r[\'question\']); k = len(texts)\n    rec = dict(dur=round(dur, 2), win=[round(s0, 2), round(s1, 2)])\n    # base + hidden states from the first option order\n    tot = np.zeros(k)\n    for n_perm, perm in enumerate(rotations(k)):\n        inp, _ = qwen_inputs(PROC, render(stem, texts, perm), media, \'av\'); inp = move_inputs(inp, MODEL_)\n        lg, store = masked_forward(inp, \'plain\', win, record_layers=LAYERS_KEEP if n_perm == 0 else None)\n        if n_perm == 0:\n            np.savez_compressed(HID / f\'h_{r["_idx"]}.npz\', last=np.stack([store[l][0].numpy() for l in LAYERS_KEEP]).astype(\'float16\'),\n                                qmean=np.stack([store[l][1].numpy() for l in LAYERS_KEEP]).astype(\'float16\'), layers=np.array(LAYERS_KEEP))\n        lp = torch.log_softmax(lg[LETTERS[:k]].cpu(), -1).numpy()\n        for pos, j in enumerate(perm): tot[j] += lp[pos]\n        del inp\n    rec[\'base\'] = tot.round(3).tolist()\n    rec[\'para1\'] = scores(media, PARA[task][0], texts)\n    rec[\'para2\'] = scores(media, PARA[task][1], texts)\n    rec[\'vision_only\'] = scores(media, stem, texts, mode=\'vision\')\n    rec[\'audio_only\'] = scores(prepare_video(blank_video(full, dur, True)), stem, texts)\n    rec[\'prior\'] = scores(prepare_video(blank_video(full, dur, False)), stem, texts)\n    rec[\'hear\'] = hear(media)\n    try: rec[\'whisper\'] = whisper_text(a, s0, s1)\n    except Exception as e: rec[\'whisper_error\'] = repr(e)[:200]\n    if task == \'CMTB\':\n        for mk in (\'gate\', \'audio_block\', \'gate_audio\'): rec[mk] = scores(media, stem, texts, mask_kind=mk, window=win)\n        rec[\'crop\'] = scores(prepare_video(cut(full, s0, s1, \'speech\')), stem, texts, mode=\'vision\')\n        g = \'ABC\'.index(r[\'answer\'])\n        if int(np.argmax(rec[\'base\'])) != g and int(np.argmax(rec[\'gate_audio\'])) == g:\n            rec[\'patching\'] = patch_sweep(media, stem, texts, win)\n    if shard_attention:\n        rec[\'attention\'] = eager_attention(media, stem, texts, win, dur)\n    return rec\n\n\ndef main():\n    global MODEL_, PROC, LETTERS\n    rows = json.loads(fetch(\'samples.json\').read_text(encoding=\'utf-8\'))\n    cells = {}\n    for i, r in enumerate(rows):\n        if r[\'data_type\'] == \'temporal\': cells.setdefault((r[\'task_type_abbr\'], r[\'answer\']), []).append((i, r))\n    items = []\n    for key in sorted(cells):\n        pool = cells[key]; random.Random(1).shuffle(pool); items += pool[:PER_LABEL]\n    items.sort(key=lambda x: x[0])\n    mine = [x for k, x in enumerate(items) if k % NSHARD == SHARD]\n    if os.environ.get(\'LAB_ONLY_IDS\'):\n        want = {int(v) for v in os.environ[\'LAB_ONLY_IDS\'].split(\',\')}; mine = [x for x in enumerate(rows) if x[0] in want]\n    res = OUT / f\'rootcause_shard{SHARD}of{NSHARD}.jsonl\'\n    done = {json.loads(l)[\'idx\'] for l in res.read_text().splitlines()} if res.exists() else set()\n    MODEL_, PROC = load_model(); LETTERS = [PROC.tokenizer.convert_tokens_to_ids(L) for L in \'ABC\']\n    print(\'model ready; items\', len(mine), \'done\', len(done), flush=True)\n    syn = OUT / \'synthetic_relation.json\'\n    if SHARD == 0 and not syn.exists() and not os.environ.get(\'LAB_NO_SYNTH\'):\n        syn.write_text(json.dumps(synthetic_relation()), encoding=\'utf-8\'); print(\'synthetic relation done\', flush=True)\n    n_att = 0\n    for n, (i, r) in enumerate(mine):\n        if i in done: continue\n        t0 = time.time(); task = r[\'task_type_abbr\']\n        rec = dict(idx=i, task=task, gold=\'ABC\'.index(r[\'answer\']))\n        try:\n            r = dict(r); r[\'_idx\'] = i\n            att = SHARD == 0 and n_att < 24\n            rec.update(run_item(task, r, att)); n_att += int(att); rec[\'sec\'] = round(time.time() - t0)\n        except Exception as e:\n            rec[\'error\'] = repr(e)[:300]\n        torch.cuda.empty_cache()\n        with res.open(\'a\') as f: f.write(json.dumps(rec) + \'\\n\')\n        if \'error\' in rec: print(n + 1, \'/\', len(mine), task, i, rec[\'error\'][:60], flush=True); continue\n        g = rec[\'gold\']; ok = lambda c: \'Y\' if c in rec and int(np.argmax(rec[c])) == g else \'-\'\n        print(n + 1, \'/\', len(mine), task, i, \'base\', ok(\'base\'), \'p1\', ok(\'para1\'), \'p2\', ok(\'para2\'), \'vis\', ok(\'vision_only\'), \'aud\', ok(\'audio_only\'),\n              \'GA\', ok(\'gate_audio\'), \'patch\' if \'patching\' in rec else \'\', rec[\'sec\'], \'s\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'job_utils.py': '"""Shared helpers for the diagnostic notebooks (exec\'d in the kaggle_rootcause.py namespace after the model is loaded).\nLabels are only ever used for scoring/analysis; nothing here feeds a label to the model."""\nimport json, random, time\nfrom pathlib import Path\nimport numpy as np, torch, soundfile as sf\n\nL4 = [PROC.tokenizer.convert_tokens_to_ids(c) for c in \'ABCD\']\nSR = 16000\n\n\n# ---------------------------------------------------------------- items\ndef sample_items(tasks, per_label, seed, max_dur=None):\n    """Balanced per (task, gold letter); round-robin across cells so a deadline cut keeps the sample balanced."""\n    rows = json.loads(fetch(\'samples.json\').read_text(encoding=\'utf-8\'))\n    cells = {}\n    for i, r in enumerate(rows):\n        if r[\'data_type\'] == \'temporal\' and r[\'task_type_abbr\'] in tasks: cells.setdefault((r[\'task_type_abbr\'], r[\'answer\']), []).append((i, r))\n    pools = []\n    for key in sorted(cells):\n        pool = cells[key]; random.Random(seed).shuffle(pool); pools.append(pool[:per_label])\n    items = []\n    for k in range(max(len(p) for p in pools)):\n        for p in pools:\n            if k < len(p): items.append(p[k])\n    return items\n\n\ndef all_temporal_items():\n    rows = json.loads(fetch(\'samples.json\').read_text(encoding=\'utf-8\'))\n    return [(i, r) for i, r in enumerate(rows) if r[\'data_type\'] == \'temporal\']\n\n\ndef prep_clip(r, max_dur=None):\n    src = cut(fetch(r[\'video_path\'].replace(BS, \'/\')), 0, 10 ** 4, \'full\')\n    a = decode_audio(src); dur = len(a) / SR\n    if max_dur is not None and dur > max_dur: raise RuntimeError(\'long\')\n    w = speech_window(a)\n    if w is None: raise RuntimeError(\'no speech activity\')\n    return src, a, dur, w\n\n\ndef speech_first(w, dur): return (w[0] + w[1]) / 2 < dur / 2\n\n\n# ---------------------------------------------------------------- scoring (up to 4 options, rotated orders summed per content)\ndef prompt_of(stem, texts, perm): return stem + \' \' + \' \'.join(f\'({L}) {texts[j]}\' for L, j in zip(\'ABCD\', perm))\n\n\ndef score(media, stem, texts, mode=\'av\', fwd=None, on_logits=None):\n    """fwd(inp) -> last-token logits (default: plain forward). on_logits(perm, logits) is called for every order."""\n    k = len(texts); tot = np.zeros(k)\n    for perm in rotations(k):\n        inp, _ = qwen_inputs(PROC, prompt_of(stem, texts, perm), media, mode); inp = move_inputs(inp, MODEL_)\n        if fwd is None:\n            with torch.no_grad(): lg = MODEL_(**inp, use_audio_in_video=(mode == \'av\')).logits[0, -1].float()\n        else: lg = fwd(inp)\n        if on_logits is not None: on_logits(perm, lg)\n        lp = torch.log_softmax(lg[L4[:k]].cpu(), -1).numpy()\n        for p_, j in enumerate(perm): tot[j] += lp[p_]\n        del inp\n    torch.cuda.empty_cache()\n    return tot.round(3).tolist()\n\n\ndef yes_minus_no(media, question, mode=\'vision\'):\n    s = score(media, question, [\'Yes\', \'No\'], mode=mode); return round(s[0] - s[1], 3)\n\n\ndef layer_out(out):\n    h = out[0] if isinstance(out, tuple) else out\n    return h[0] if h.dim() == 3 else h\n\n\ndef forward_record(inp, layers, groups, use_audio=True):\n    """plain forward; for each recorded layer (1-based output index) the mean hidden state of every boolean token group."""\n    store, hooks = {}, []\n    for li in layers:\n        def hook(mod, args, out, li=li):\n            h = layer_out(out)\n            store[li] = {g: h[m.to(h.device)].float().mean(0).cpu() for g, m in groups.items() if bool(m.any())}\n        hooks.append(MODEL_.model.layers[li - 1].register_forward_hook(hook))\n    try:\n        with torch.no_grad(): lg = MODEL_(**inp, use_audio_in_video=use_audio).logits[0, -1].float()\n    finally:\n        for h in hooks: h.remove()\n    return lg, store\n\n\ndef token_groups(inp, dur, win):\n    pos, sec, is_v, is_a, last_media = media_layout(MODEL_, PROC, inp)\n    sec, is_v, is_a = sec.cpu(), is_v.cpu(), is_a.cpu()\n    L = inp[\'input_ids\'].shape[1]; idx = torch.arange(L)\n    inside = (sec >= win[0] - 1.0) & (sec <= win[1] + 1.0)\n    g = {\'last\': idx == L - 1, \'q\': idx > last_media,\n         \'v_first\': is_v & (sec < dur / 2), \'v_second\': is_v & (sec >= dur / 2),\n         \'a_speech\': is_a & inside, \'a_rest\': is_a & ~inside}\n    return g, (pos, sec, is_v, is_a, last_media)\n\n\n# ---------------------------------------------------------------- clip variants (ffmpeg, cached in CLIPS)\ndef with_audio(src, audio, tag):\n    w = CLIPS / f\'{Path(src).stem}_{tag}.wav\'; sf.write(w, audio, SR)\n    dst = CLIPS / f\'{Path(src).stem}_{tag}.mp4\'\n    if not dst.exists(): ffmpeg(\'-i\', str(src), \'-i\', str(w), \'-map\', \'0:v:0\', \'-map\', \'1:a:0\', \'-c:v\', \'copy\', \'-c:a\', \'aac\', \'-shortest\', str(dst))\n    return dst\n\n\ndef concat_clips(parts, tag, audio_from=None):\n    dst = CLIPS / f\'{Path(parts[0]).stem}_{tag}.mp4\'\n    if dst.exists(): return dst\n    ins = []\n    for p in parts: ins += [\'-i\', str(p)]\n    n = len(parts)\n    if audio_from is None:\n        fc = \'\'.join(f\'[{k}:v:0][{k}:a:0]\' for k in range(n)) + f\'concat=n={n}:v=1:a=1[v][a]\'\n        ffmpeg(*ins, \'-filter_complex\', fc, \'-map\', \'[v]\', \'-map\', \'[a]\', *ENC, str(dst))\n    else:\n        fc = \'\'.join(f\'[{k}:v:0]\' for k in range(n)) + f\'concat=n={n}:v=1:a=0[v]\'\n        ffmpeg(*ins, \'-i\', str(audio_from), \'-filter_complex\', fc, \'-map\', \'[v]\', \'-map\', f\'{n}:a:0\', *ENC, \'-shortest\', str(dst))\n    return dst\n\n\ndef halves(src, dur):\n    return cut(src, 0, dur / 2, \'h1\'), cut(src, dur / 2, dur, \'h2\')\n\n\ndef pink(n, seed):\n    X = np.fft.rfft(np.random.default_rng(seed).standard_normal(n)); f = np.arange(len(X), dtype=\'float64\'); f[0] = 1\n    x = np.fft.irfft(X / np.sqrt(f), n); return (x / (x.std() + 1e-12)).astype(\'float32\')\n\n\ndef noisy_audio(a, w, snr_db=10.0, seed=0):\n    """pink noise over the whole track at snr_db below the speech RMS: removes the benchmark\'s digital silence."""\n    s = a[int(w[0] * SR): int(w[1] * SR)]; ps = float(np.sqrt(np.mean(s ** 2) + 1e-12))\n    return np.clip(a + pink(len(a), seed) * ps * 10 ** (-snr_db / 20), -1, 1).astype(\'float32\')\n\n\n# ---------------------------------------------------------------- loop with resume + deadline\ndef run_loop(items, out_name, fn, show=(), min_left=1200):\n    items = items[:int(globals().get(\'LIMIT\', 10 ** 9))]          # LIMIT is set only by the smoke test\n    res = OUT / out_name\n    done = {json.loads(l)[\'idx\'] for l in res.read_text().splitlines() if l.strip()} if res.exists() else set()\n    print(\'items\', len(items), \'already done\', len(done), flush=True)\n    group_stats = {}\n    for n, (i, r) in enumerate(items):\n        if i in done: continue\n        if time_left() < min_left: print(\'deadline reached: stopping cleanly\', flush=True); break\n        t0 = time.time(); rec = dict(idx=i, task=r[\'task_type_abbr\'], gold=\'ABCD\'.index(r[\'answer\']))\n        try:\n            rec.update(fn(i, r)); rec[\'sec\'] = round(time.time() - t0)\n        except torch.cuda.OutOfMemoryError:\n            rec[\'error\'] = \'OOM\'\n        except Exception as e:\n            rec[\'error\'] = repr(e)[:300]\n        torch.cuda.empty_cache()\n        with res.open(\'a\') as f: f.write(json.dumps(rec) + \'\\n\')\n        if \'error\' in rec: print(n + 1, len(items), rec[\'task\'], i, \'ERR\', rec[\'error\'][:80], flush=True); continue\n        g = rec[\'gold\']\n        marks = \' \'.join(f\'{c}={"Y" if int(np.argmax(rec[c])) == g else "-"}\' for c in show if isinstance(rec.get(c), list))\n        print(n + 1, len(items), rec[\'task\'], i, marks, rec[\'sec\'], \'s\', flush=True)\n        if globals().get(\'PROGRESS_GROUPS\', False):\n            key = (rec[\'task\'], int(g))\n            z = group_stats.setdefault(key, {c: [0, 0] for c in show})\n            for c in show:\n                if isinstance(rec.get(c), list):\n                    z[c][1] += 1\n                    z[c][0] += int(np.argmax(rec[c]) == g)\n            parts = []\n            for (tt, gg), zz in sorted(group_stats.items()):\n                parts.append(f\'{tt}/{chr(65 + gg)} \' + \' \'.join(f\'{c}={zz[c][0]}/{zz[c][1]}\' for c in show if c in zz))\n            print(\'GROUP_PROGRESS \' + \' | \'.join(parts), flush=True)\n    return [json.loads(l) for l in res.read_text().splitlines() if l.strip()] if res.exists() else []\n\n\ndef acc(R, cond, task=None, expect=None):\n    """accuracy (%) and n of condition `cond`; expect(rec) gives the expected index (default: gold)."""\n    v = [int(np.argmax(r[cond])) == (expect(r) if expect else r[\'gold\']) for r in R\n         if \'error\' not in r and isinstance(r.get(cond), list) and (task is None or r[\'task\'] == task) and (expect is None or expect(r) is not None)]\n    return (round(100 * float(np.mean(v)), 1) if v else None, len(v))\n', 'nb_runner.py': '"""Load Qwen2.5-Omni-3B once into the namespace of kaggle_rootcause.py, then exec job_utils.py and one job file in it.\nUsage: python nb_runner.py ROOT OUT JOB.py \'{"PARAM": value, "HOURS": 10.5}\'\nThe job sees every helper of kaggle_rootcause.py (cut, fetch, scores, masked_forward, hear, whisper_text, ...), the model\n(MODEL_, PROC, LETTERS), OUT, the params as globals, and time_left() (seconds before the self-imposed deadline)."""\nimport sys, os, time, json\nfrom pathlib import Path\n\nROOT, OUT, JOB = Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3])\nPARAMS = json.loads(sys.argv[4]) if len(sys.argv) > 4 else {}\nt_start = time.time()\nsys.argv = [\'kaggle_rootcause.py\', str(ROOT), str(OUT), \'1\', \'0\', \'1\']\nNS = {\'__name__\': \'rc_namespace\', \'__file__\': str(ROOT / \'kaggle_rootcause.py\')}\nexec(compile((ROOT / \'kaggle_rootcause.py\').read_text(encoding=\'utf-8\'), \'kaggle_rootcause.py\', \'exec\'), NS)\nNS[\'MODEL_\'], NS[\'PROC\'] = NS[\'load_model\']()\nNS[\'LETTERS\'] = [NS[\'PROC\'].tokenizer.convert_tokens_to_ids(L) for L in \'ABC\']\nhours = float(PARAMS.pop(\'HOURS\', 10.5))\nNS.update(OUT=OUT, ROOT=ROOT, DEADLINE=t_start + hours * 3600, **PARAMS)\nNS[\'time_left\'] = lambda: NS[\'DEADLINE\'] - time.time()\nprint(\'model ready in\', round(time.time() - t_start), \'s; params\', PARAMS, \'; deadline in\', hours, \'h\', flush=True)\nexec(compile((ROOT / \'job_utils.py\').read_text(encoding=\'utf-8\'), \'job_utils.py\', \'exec\'), NS)\nexec(compile(JOB.read_text(encoding=\'utf-8\'), JOB.name, \'exec\'), NS)\nprint(\'RESULT\', json.dumps(NS.get(\'RESULT\'), default=str, indent=1), flush=True)\n', 'sem_chrono.py': '"""SEM-CHRONO — the friend\'s ChronoDeSmear (Round 5, R5_C0..C6) evaluated UNCHANGED on the SVHalluc SEMANTIC tasks, inside one Kaggle notebook.\nProtocol fixed beforehand: results/semantic_chrono/protocol.md.   Needs the files written by the notebook (see `extra` in build_notebooks.py):\nchronodesmear_cache40.py, chronodesmear40.py, chronodesmear_math.py, avbind_sdpa_compat.py, chronodesmear_proposal_original.py, semantic_manifest_job.py.\nSteps: (1) copy the helpers to OUT/uploads and patch the friend\'s cache/runner by text substitution (manifest, cache and output paths only);\n(2) build the balanced manifest (GSA/FGSA/CSMB x Yes/No = 6 cells, SEM_PER_CELL each, clip <= SEM_MAXDUR s, seed SEM_SEED);\n(3) encode every answer rotation once (cache, FP16 inputs_embeds); (4) run R5_C0..C6 on this shard (SEM_SHARD 0 or 1 = half of every cell).\nParameters (globals): SEM_SHARD, SEM_PER_CELL, SEM_SEED, SEM_MAXDUR; LIMIT (set by the smoke test) -> 1 sample per cell, 2 samples run.\nWrites OUT/sem_chrono_s<shard+1>.jsonl (+ OUT/semantic_chrono/*: per-rotation JSON, errors) ."""\nimport json, re, shutil\n\nSEM_SHARD = int(globals().get(\'SEM_SHARD\', 0))\nSMOKE = globals().get(\'LIMIT\') is not None\nSEM_PER_CELL = 1 if SMOKE else int(globals().get(\'SEM_PER_CELL\', 10))\nSEM_SEED = int(globals().get(\'SEM_SEED\', 31337)); SEM_MAXDUR = float(globals().get(\'SEM_MAXDUR\', 40.0))\nU = OUT / \'uploads\'; U.mkdir(parents=True, exist_ok=True)\n\n# (1) helpers + patched copies of the friend\'s cache and runner (only paths change)\nfor name in (\'chronodesmear_math.py\', \'avbind_sdpa_compat.py\', \'chronodesmear_proposal_original.py\'):\n    (U / name).write_text((ROOT / name).read_text(encoding=\'utf-8\'), encoding=\'utf-8\')\ncache = (ROOT / \'chronodesmear_cache40.py\').read_text(encoding=\'utf-8\')\ncache = cache.replace(\'research40_manifest.json\', \'semantic_manifest.json\')\ncache = re.sub(r"\\n\\s*assert CD_MANIFEST\\[\'total\'\\]==40\\n", \'\\n\', cache)\ncache = cache.replace(\'argon_latent40/decoder_cache\', \'semantic_chrono/decoder_cache\')\nrun = (ROOT / \'chronodesmear40.py\').read_text(encoding=\'utf-8\')\nrun = run.replace(\'uploads/chronodesmear_cache40.py\', \'uploads/semantic_cache.py\').replace(\'uploads/chronodesmear40.py\', \'uploads/semantic_chrono.py\')\nrun = run.replace("OUT/\'chronodesmear40\'", "OUT/\'semantic_chrono\'")\nassert "assert CD_MANIFEST[\'total\']==40" not in cache and \'semantic_manifest.json\' in cache and "OUT/\'semantic_chrono\'" in run\n(U / \'semantic_cache.py\').write_text(cache, encoding=\'utf-8\'); (U / \'semantic_chrono.py\').write_text(run, encoding=\'utf-8\')\n\n# (2) balanced semantic manifest (deterministic: seed + durations only)\nexec(compile((ROOT / \'semantic_manifest_job.py\').read_text(encoding=\'utf-8\'), \'semantic_manifest_job.py\', \'exec\'), globals())\nCD_SERVER = 1 if SMOKE else SEM_SHARD + 1\nif SMOKE:                                                    # smoke: 2 samples only\n    man = json.loads((U / \'semantic_manifest.json\').read_text()); man[\'server1\'] = man[\'server1\'][:2]\n    (U / \'semantic_manifest.json\').write_text(json.dumps(man), encoding=\'utf-8\')\nprint(\'shard\', SEM_SHARD, \'server list size\', len(json.loads((U / \'semantic_manifest.json\').read_text())[f\'server{CD_SERVER}\']), flush=True)\n\n# (3) encode every rotation once\nCD_LIMIT = None\nCD_CACHE_RUN = True\nexec(compile((U / \'semantic_cache.py\').read_text(), \'semantic_cache.py\', \'exec\'), globals())\n\n# (4) run the seven configurations\nexec(compile((U / \'semantic_chrono.py\').read_text(), \'semantic_chrono.py\', \'exec\'), globals())\nsrc = OUT / \'semantic_chrono\' / f\'results_s{CD_SERVER}.jsonl\'\nif src.exists(): shutil.copy(src, OUT / f\'sem_chrono_s{CD_SERVER}.jsonl\')\n', 'chronodesmear_cache40.py': '"""Encode the unchanged frozen40 inputs once; reusable by all decoder audits."""\nimport gc, hashlib, json, time\nimport numpy as np\nimport torch\nfrom pathlib import Path\n\nCD_SERVER = globals().get(\'CD_SERVER\', globals().get(\'R40_SERVER\', 1))\nCD_MANIFEST = json.loads((OUT/\'uploads/research40_manifest.json\').read_text())\nCD_SHA = CD_MANIFEST[\'manifest_sha256\']\nCD_CACHE = OUT/\'argon_latent40/decoder_cache\'\nCD_CACHE.mkdir(parents=True, exist_ok=True)\nCD_LABELS = [PROC.tokenizer.convert_tokens_to_ids(c) for c in \'ABC\']\nexec(compile((OUT/\'uploads/avbind_sdpa_compat.py\').read_text(),\'sdpa_compat\',\'exec\'),globals())\n\ndef cd_prompt(stem, texts, perm):\n    return render(stem,texts,perm)+\'\\nAnswer with only the letter of the best answer.\'\n\ndef cd_encode(inp, k):\n    captured={}\n    def capture(_module,args,kwargs):\n        for key in (\'inputs_embeds\',\'position_ids\',\'attention_mask\',\'cache_position\'):\n            if kwargs.get(key) is not None: captured[key]=kwargs[key].detach().cpu()\n    handles=[MODEL_.model.register_forward_pre_hook(capture,with_kwargs=True),\n             MODEL_.lm_head.register_forward_pre_hook(lambda _m,args:(args[0][:,-1:,:],))]\n    try:\n        with torch.inference_mode():\n            logits=MODEL_(**inp,use_audio_in_video=True,use_cache=False).logits[0,-1].float().cpu()\n    finally:\n        for handle in handles: handle.remove()\n    _,sec,is_v,is_a,last=media_layout(MODEL_,PROC,inp)\n    spg=inp.get(\'video_second_per_grid\')\n    assert torch.isfinite(logits).all()\n    return dict(decoder=captured,native_logits=logits[CD_LABELS[:k]],sec=sec.cpu(),\n        is_v=is_v.cpu(),is_a=is_a.cpu(),last_media=int(last),\n        spg=float(spg.reshape(-1)[0]) if spg is not None else None,\n        tokens=int(inp[\'input_ids\'].shape[1]),manifest_sha256=CD_SHA,prompt=\'letter_only_v1\')\n\ndef cd_build_cache():\n    assert CD_MANIFEST[\'total\']==40\n    completed=0\n    for sample in CD_MANIFEST[f\'server{CD_SERVER}\']:\n        assert sample[\'duration\']<=45 and sample[\'decoded_audio_duration\']<=45\n        stem,texts=parse_question(sample[\'question\']);k=len(texts);media=None\n        for rotation,perm in enumerate(rotations(k)):\n            cache=CD_CACHE/f"{sample[\'idx\']}_rot{rotation}.pt"\n            if not cache.exists():\n                if media is None:\n                    source=fetch(sample[\'video_path\'])\n                    media=prepare_video(source,max_pixels=int(160*min(1.,32/sample[\'duration\']))*28*28)\n                inputs,_=qwen_inputs(PROC,cd_prompt(stem,texts,perm),media,\'av\')\n                inputs=move_inputs(inputs,MODEL_)\n                payload=cd_encode(inputs,k)\n                tmp=cache.with_suffix(\'.tmp\');torch.save(payload,tmp);tmp.replace(cache)\n                del inputs,payload;gc.collect();torch.cuda.empty_cache()\n            completed+=1\n        del media\n        print(\'CHRONO_CACHE\',CD_SERVER,sample[\'idx\'],completed,flush=True)\n    return dict(server=CD_SERVER,rotations=completed,manifest_sha256=CD_SHA)\n\nif globals().get(\'CD_CACHE_RUN\',True):\n    RESULT=cd_build_cache()\n    print(\'CHRONO_CACHE_COMPLETE\',json.dumps(RESULT),flush=True)\n', 'chronodesmear40.py': '"""Round5 proposal mathematics, frozen40 harness and explicitly corrected boundaries."""\nimport ast,gc,hashlib,json,time,traceback\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nCD_SERVER=globals().get(\'CD_SERVER\',1)\nCD_LIMIT=globals().get(\'CD_LIMIT\')\nCD_CACHE_RUN=False\nexec(compile((OUT/\'uploads/chronodesmear_cache40.py\').read_text(),\'cache_definitions\',\'exec\'),globals())\nexec(compile((OUT/\'uploads/chronodesmear_math.py\').read_text(),\'chrono_math\',\'exec\'),globals())\nCD_DIR=OUT/\'chronodesmear40\';CD_DIR.mkdir(exist_ok=True)\nCD_ORIGINAL=(OUT/\'uploads/chronodesmear_proposal_original.py\').read_text()\nCD_PROPOSAL_SHA=hashlib.sha256(CD_ORIGINAL.encode()).hexdigest()\nCD_RUNTIME_SHA=hashlib.sha256((OUT/\'uploads/chronodesmear40.py\').read_bytes()\n    +(OUT/\'uploads/chronodesmear_math.py\').read_bytes()).hexdigest()\n_wanted={\'hidden_to_manifold_thought\',\'detect_active_window\',\'center_option_slice\',\n         \'compute_av_resonance_and_shift_diagnostics\',\'make_shuffled_delta\'}\n_parts=[n for n in ast.parse(CD_ORIGINAL).body if isinstance(n,ast.FunctionDef) and n.name in _wanted]\nassert len(_parts)==len(_wanted)\nexec(compile(ast.Module(body=_parts,type_ignores=[]),\'original_round5_helpers\',\'exec\'),globals())\nCD_METHODS=[\'R5_C0_baseline\',\'R5_C1_causal_desmear_1p\',\'R5_C2_stratified_dyn_cd\',\n    \'R5_C3_balanced_modal_cd\',\'R5_C4_stem_plus_media\',\'R5_C5_chronodesmear_full\',\n    \'R5_C6_multiseed_shuffled\']\nCD_REC={10:.24,12:.30,14:.30,16:.24}\nCD_DS_LAYERS=(8,10,12,14,16)\n\ndef cd_find_span(payload,stem):\n    """Exact lexical embedding matches, replacing the proposal\'s token-count guess."""\n    start=payload[\'last_media\']+1\n    tail=payload[\'decoder\'][\'inputs_embeds\'][0,start:].cpu()\n    device=MODEL_.get_input_embeddings().weight.device\n    def find(texts,minimum=0):\n        matches=[]\n        for text in texts:\n            ids=PROC.tokenizer.encode(text,add_special_tokens=False)\n            with torch.inference_mode():\n                embeds=MODEL_.get_input_embeddings()(torch.tensor(ids,device=device)).cpu()\n            for offset in range(minimum,len(tail)-len(ids)+1):\n                if torch.equal(tail[offset:offset+len(ids)],embeds): matches.append((offset,offset+len(ids)))\n        if not matches: raise ValueError((\'Could not exactly locate text boundary\',texts))\n        return min(matches)\n    stem_start,stem_end=find((stem,\' \'+stem,\'\\n\'+stem))\n    option_start,_=find((\'(A)\',\' (A)\',\'\\n(A)\'),minimum=stem_end)\n    assert 0<=stem_start<stem_end<=option_start<len(tail)\n    return dict(span_start=start,option_start=start+option_start,\n        stem_len=option_start,lexical_stem_start=start+stem_start,lexical_stem_end=start+stem_end)\n\ndef cd_forward(payload,mask=None,thought=None,delta=None,span_end=None,\n               recurrent_layers=None,desmear=False,capture=False,plans=None,\n               strengths=(.26,.18),record_layers=(22,)):\n    device=MODEL_.get_input_embeddings().weight.device\n    kwargs={k:v.to(device) for k,v in payload[\'decoder\'].items()}\n    start=payload[\'last_media\']+1;end=payload[\'tokens\'] if span_end is None else span_end\n    rec=CD_REC if recurrent_layers is None else recurrent_layers\n    handles=[];masks={};captured={}\n    def pre(module,args,kw):\n        h=kw.get(\'hidden_states\',args[0] if args else None);key=str(h.device)\n        if key not in masks: masks[key]=mask.to(h.device)\n        kw[\'attention_mask\']=masks[key]\n        return args,kw\n    def post(layer):\n        def hook(module,args,out):\n            h=out[0] if isinstance(out,tuple) else out\n            changed=False\n            if thought is not None and layer in (0,1,2):\n                h=h.clone();changed=True;q=h[0,start:end].float()\n                h[0,start:end]=(q+.18*q.norm(dim=-1,keepdim=True).mean().clamp_min(1e-5)\n                    *thought.to(h.device)[None]).to(h.dtype)\n            if desmear and layer in CD_DS_LAYERS:\n                if not changed: h=h.clone();changed=True\n                for plan,strength in zip(plans,strengths):\n                    if not len(plan[\'indices\']): continue\n                    indices=plan[\'indices\'].to(h.device)\n                    h[0,indices]=desmear_values(h[0,indices].float(),plan,strength).to(h.dtype)\n            if delta is not None and layer in rec:\n                if not changed: h=h.clone();changed=True\n                q=h[0,start:end].float();d=delta.to(h.device)\n                assert q.shape==d.shape,(q.shape,d.shape)\n                h[0,start:end]=(q+rec[layer]*q.norm(dim=-1,keepdim=True).clamp_min(1e-5)\n                    /d.norm(dim=-1,keepdim=True).clamp_min(1e-5)*d).to(h.dtype)\n            if capture and layer in record_layers: captured[layer]=h[0].detach().float().cpu()\n            return (h,)+out[1:] if isinstance(out,tuple) else h\n        return hook\n    try:\n        for layer,block in enumerate(MODEL_.model.layers):\n            if mask is not None: handles.append(block.register_forward_pre_hook(pre,with_kwargs=True))\n            if (thought is not None and layer in (0,1,2)) or (desmear and layer in CD_DS_LAYERS) or (\n                delta is not None and layer in rec) or (capture and layer in record_layers):\n                handles.append(block.register_forward_hook(post(layer)))\n        with torch.inference_mode():\n            output=MODEL_.model(**kwargs,use_cache=False,return_dict=True)\n            logits=MODEL_.lm_head(output.last_hidden_state[:,-1:,:].to(MODEL_.lm_head.weight.device))[0,-1].float().cpu()\n    finally:\n        for handle in handles: handle.remove()\n        masks.clear()\n    assert torch.isfinite(logits).all()\n    return logits,captured\n\ndef cd_rotation(payload,audio,duration,k,stem):\n    started=time.time();last=payload[\'last_media\'];v=payload[\'is_v\'];a=payload[\'is_a\'];sec=payload[\'sec\']\n    dtype=MODEL_.get_input_embeddings().weight.dtype\n    plans=[interval_plan(torch.where(source)[0],sec) for source in (v,a)]\n    span=cd_find_span(payload,stem)\n    native,cap=cd_forward(payload,capture=True)\n    replay=float((native[CD_LABELS[:k]]-payload[\'native_logits\']).abs().max())\n    assert replay<=.12,(\'Native replay\',replay)\n    base=cap[22];base_q=base[last+1:]\n    static,innovation,v_indices,bin_counts=static_visual_mask(base,v,sec,duration)\n    thought,resonance=compute_av_resonance_and_shift_diagnostics(base,v,a,static,sec,duration)\n    del cap\n    ds,cap=cd_forward(payload,desmear=True,plans=plans,capture=True)\n    ds_q=cap[22][last+1:];del cap\n    window=detect_active_window(audio,duration)\n    branch={}\n    for mode in (\'static_plus_aud\',\'v_all_only\',\'hard_sync\',\'anti_sync\'):\n        mask=counterfactual_mask(payload,mode,dtype,static,window)\n        _,cap=cd_forward(payload,mask=mask,capture=True)\n        branch[mode]=cap[22][last+1:].clone()\n        del mask,cap;torch.cuda.empty_cache()\n    # Exact supplied code, including ds and V-only terms absent from the prose formula.\n    delta_raw=(.5*(ds_q+base_q)-branch[\'static_plus_aud\'])+.35*(\n        branch[\'v_all_only\']-branch[\'static_plus_aud\'])\n    centered=center_option_slice(delta_raw,span[\'stem_len\'])\n    balanced=.5*centered+.5*center_option_slice(.5*(branch[\'v_all_only\']+\n        branch[\'hard_sync\'])-branch[\'anti_sync\'],span[\'stem_len\'])\n    logits={\'R5_C0_baseline\':native,\'R5_C1_causal_desmear_1p\':ds}\n    logits[\'R5_C2_stratified_dyn_cd\'],_=cd_forward(payload,delta=centered)\n    logits[\'R5_C3_balanced_modal_cd\'],_=cd_forward(payload,delta=balanced,desmear=True,plans=plans)\n    logits[\'R5_C4_stem_plus_media\'],_=cd_forward(payload,delta=delta_raw[:span[\'stem_len\']],\n        span_end=span[\'option_start\'],recurrent_layers={10:.28,12:.35,14:.35,16:.28},\n        desmear=True,plans=plans)\n    logits[\'R5_C5_chronodesmear_full\'],_=cd_forward(payload,thought=thought,delta=centered,\n        desmear=True,plans=plans)\n    scores={m:torch.log_softmax(lg[CD_LABELS[:k]],dim=-1).numpy().tolist() for m,lg in logits.items()}\n    seed_scores={};norm_errors={}\n    for seed in (42,1337,2027):\n        shuffled=make_shuffled_delta(centered,seed)\n        norm_errors[str(seed)]=float((shuffled.norm(dim=-1)-centered.norm(dim=-1)).abs().max())\n        assert torch.allclose(shuffled.norm(dim=-1),centered.norm(dim=-1),rtol=1e-5,atol=1e-5)\n        lg,_=cd_forward(payload,delta=shuffled)\n        seed_scores[str(seed)]=torch.log_softmax(lg[CD_LABELS[:k]],dim=-1).numpy().tolist()\n    scores[\'R5_C6_multiseed_shuffled\']=np.mean(list(seed_scores.values()),axis=0).tolist()\n    del base,ds_q,base_q,branch,centered,balanced,delta_raw,logits\n    return dict(scores=scores,shuffled_seed_scores=seed_scores,replay_error=replay,tokens=payload[\'tokens\'],\n        span=span,static_bins=bin_counts,static_count=int(static.sum()),visual_count=int(v.sum()),\n        av_resonance_EML=resonance,resonance_is_event_grounding=False,\n        audio_window_for_C3_only=list(window),shuffle_norm_errors=norm_errors,\n        echo_valid_counts=[int(p[\'has_previous\'].sum()) for p in plans],seconds=time.time()-started)\n\ndef cd_run():\n    samples=CD_MANIFEST[f\'server{CD_SERVER}\']\n    if CD_LIMIT: samples=samples[:CD_LIMIT]\n    output=CD_DIR/f\'results_s{CD_SERVER}.jsonl\'\n    done={json.loads(line)[\'idx\'] for line in output.read_text().splitlines() if line.strip()} if output.exists() else set()\n    for sample in samples:\n        idx=sample[\'idx\']\n        if idx in done: continue\n        assert sample[\'duration\']<=45 and sample[\'decoded_audio_duration\']<=45\n        started=time.time();stem,texts=parse_question(sample[\'question\']);k=len(texts)\n        audio=decode_audio(fetch(sample[\'video_path\']))\n        assert abs(len(audio)/16000-sample[\'decoded_audio_duration\'])<.2\n        totals={m:np.zeros(k) for m in CD_METHODS};seeds={str(s):np.zeros(k) for s in (42,1337,2027)};audits=[]\n        for rotation,perm in enumerate(rotations(k)):\n            path=CD_DIR/f\'{idx}_rot{rotation}.json\'\n            if path.exists(): result=json.loads(path.read_text())\n            else:\n                print(\'CHRONO_ROT_START\',CD_SERVER,idx,rotation,flush=True)\n                payload=torch.load(CD_CACHE/f\'{idx}_rot{rotation}.pt\',map_location=\'cpu\',weights_only=False)\n                assert payload[\'manifest_sha256\']==CD_SHA\n                result=cd_rotation(payload,audio,sample[\'duration\'],k,stem)\n                result.update(idx=idx,rotation=rotation,perm=list(perm),manifest_sha256=CD_SHA,\n                    proposal_sha256=CD_PROPOSAL_SHA,runtime_sha256=CD_RUNTIME_SHA)\n                tmp=path.with_suffix(\'.tmp\');tmp.write_text(json.dumps(result));tmp.replace(path)\n                del payload;gc.collect();torch.cuda.empty_cache()\n            assert result[\'manifest_sha256\']==CD_SHA and result[\'runtime_sha256\']==CD_RUNTIME_SHA\n            for method,values in result[\'scores\'].items():\n                for slot,option in enumerate(perm): totals[method][option]+=values[slot]\n            for seed,values in result[\'shuffled_seed_scores\'].items():\n                for slot,option in enumerate(perm): seeds[seed][option]+=values[slot]\n            audits.append({key:value for key,value in result.items() if key not in (\'scores\',\'shuffled_seed_scores\')})\n        record=dict(idx=idx,task=sample[\'task\'],gold=sample[\'gold\'],status=\'ok\',manifest_sha256=CD_SHA,\n            runtime_sha256=CD_RUNTIME_SHA,proposal_sha256=CD_PROPOSAL_SHA,\n            scores={m:values.tolist() for m,values in totals.items()},\n            predictions={m:chr(65+int(values.argmax())) for m,values in totals.items()},\n            shuffled_seed_scores={s:values.tolist() for s,values in seeds.items()},audit=audits,\n            seconds=round(time.time()-started,2))\n        with output.open(\'a\') as handle: handle.write(json.dumps(record)+\'\\n\')\n        done.add(idx)\n        print(\'CHRONO_SAMPLE\',len(done),\'/\',len(samples),json.dumps({k:v for k,v in record.items() if k!=\'audit\'}),flush=True)\n    return dict(server=CD_SERVER,completed=len(done),expected=len(samples),manifest_sha256=CD_SHA)\n\ntry: RESULT=cd_run()\nexcept Exception:\n    with (CD_DIR/f\'errors_s{CD_SERVER}.jsonl\').open(\'a\') as handle:\n        handle.write(json.dumps(dict(error=traceback.format_exc()))+\'\\n\')\n    raise\nprint(\'CHRONO_COMPLETE\',json.dumps(RESULT),flush=True)\n', 'chronodesmear_math.py': '"""Bounded-memory equivalents of Round5 temporal averages; no model globals."""\nimport torch\nimport torch.nn.functional as F\n\ndef interval_plan(indices, seconds, min_lag=1.8, max_lag=12.0):\n    indices=indices.cpu().long();times=seconds.cpu().float()[indices]\n    assert bool((times[1:]>=times[:-1]).all()), \'Source timestamps must be monotone\'\n    # searchsorted boundaries implement inclusive dt >= min_lag and <= max_lag.\n    # Correct one-ULP subtraction boundary differences with explicit comparisons.\n    low=torch.searchsorted(times,times-max_lag,right=False)\n    high=torch.searchsorted(times,times-min_lag,right=True)\n    for row in range(len(times)):\n        while low[row]>0 and times[row]-times[low[row]-1]<=max_lag: low[row]-=1\n        while low[row]<len(times) and times[row]-times[low[row]]>max_lag: low[row]+=1\n        while high[row]>0 and times[row]-times[high[row]-1]<min_lag: high[row]-=1\n        while high[row]<len(times) and times[row]-times[high[row]]>=min_lag: high[row]+=1\n    return dict(indices=indices,low=low,high=high,has_previous=high>low)\n\ndef interval_mean(values, plan):\n    prefix=torch.cat((torch.zeros_like(values[:1]),values.cumsum(0)),dim=0)\n    low=plan[\'low\'].to(values.device);high=plan[\'high\'].to(values.device)\n    count=(high-low).clamp_min(1).unsqueeze(-1)\n    return (prefix[high]-prefix[low])/count\n\ndef desmear_values(values,plan,strength):\n    if strength==0: return values\n    valid=plan[\'has_previous\'].to(values.device)\n    if not bool(valid.any()): return values\n    echo=interval_mean(values,plan)\n    cleaned=values-strength*valid[:,None]*echo\n    cleaned=cleaned*(values.norm(dim=-1,keepdim=True).clamp_min(1e-5)\n        /cleaned.norm(dim=-1,keepdim=True).clamp_min(1e-5))\n    # Exact identity for tokens without a past, rather than numerical norm roundoff.\n    return torch.where(valid[:,None],cleaned,values)\n\ndef static_visual_mask(states,is_v,seconds,duration,n_bins=5,static_frac=.45):\n    indices=torch.where(is_v.cpu().bool())[0]\n    if not len(indices): raise ValueError(\'No visual tokens\')\n    vectors=F.normalize(states[indices].float(),dim=-1)\n    mean=F.normalize(vectors.mean(0,keepdim=True),dim=-1)\n    global_change=1-(vectors*mean).sum(-1)\n    # Original local predicate is strict dt > .5 and inclusive dt <= 3.\n    times=seconds.cpu().float()[indices]\n    plan=interval_plan(indices,seconds,.5,3.)\n    for row in range(len(times)):\n        while plan[\'high\'][row]>0 and times[row]-times[plan[\'high\'][row]-1]<=.5:\n            plan[\'high\'][row]-=1\n    plan[\'has_previous\']=plan[\'high\']>plan[\'low\']\n    local_mean=F.normalize(interval_mean(vectors,plan),dim=-1)\n    local_change=torch.where(plan[\'has_previous\'],1-(vectors*local_mean).sum(-1),global_change)\n    innovation=.55*global_change+.45*local_change\n    selected=torch.zeros(len(is_v),dtype=torch.bool)\n    lower,upper=float(times.min()),float(times.max());span=max(1e-3,upper-lower)\n    bin_counts=[]\n    for b in range(n_bins):\n        left=lower+b/n_bins*span\n        right=lower+(b+1)/n_bins*span+(1e-4 if b==n_bins-1 else 0.)\n        members=torch.where((times>=left)&(times<right))[0]\n        count=max(1,int(round(static_frac*len(members)))) if len(members) else 0\n        if count:\n            chosen=members[torch.topk(innovation[members],count,largest=False).indices]\n            selected[indices[chosen]]=True\n        bin_counts.append(dict(bin=b,total=len(members),static=count))\n    return selected,innovation,indices,bin_counts\n\ndef counterfactual_mask(payload,mode,dtype,static=None,window=None):\n    length=payload[\'tokens\'];v=payload[\'is_v\'].bool();a=payload[\'is_a\'].bool()\n    mask=torch.zeros((length,length),dtype=dtype)\n    mask.masked_fill_(torch.ones((length,length),dtype=torch.bool).triu_(1),torch.finfo(dtype).min)\n    if mode==\'static_plus_aud\': blocked=v&~static\n    elif mode==\'v_all_only\': blocked=a\n    elif mode in (\'hard_sync\',\'anti_sync\'):\n        start,end=window;sec=payload[\'sec\'].float();width=payload[\'spg\'] or 0.\n        inside=v&((sec+width)>start)&(sec<end)\n        if not bool(inside.any()): inside=v.clone()\n        blocked=v&~inside if mode==\'hard_sync\' else inside\n    else: raise ValueError(mode)\n    mask[(~blocked)[:,None]&blocked[None,:]]=torch.finfo(dtype).min\n    if mode==\'hard_sync\': mask[(~a)[:,None]&a[None,:]]=torch.finfo(dtype).min\n    return mask[None,None]\n', 'avbind_sdpa_compat.py': '"""T4 memory fix: explicit KV repeats instead of unsupported fused GQA.\n\nNo media reduction, mask/position edits, or attention-score modification.\nValidate the algebra on the GPU before enabling the compatibility path.\n"""\nimport transformers.integrations.sdpa_attention as sdpa_integration\nif not globals().get(\'AVB_SDPA_COMPAT\'):\n    if not hasattr(sdpa_integration,\'use_gqa_in_sdpa\'):\n        raise RuntimeError(\'Unrecognized Transformers SDPA implementation\')\n    from torch.nn.attention import sdpa_kernel,SDPBackend\n    q=torch.randn(1,8,128,64,device=\'cuda:0\',dtype=torch.float16)\n    k=torch.randn(1,2,128,64,device=q.device,dtype=q.dtype);v=torch.randn_like(k)\n    with sdpa_kernel(SDPBackend.MATH):\n        reference=torch.nn.functional.scaled_dot_product_attention(q,k,v,enable_gqa=True,is_causal=True)\n    with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):\n        candidate=torch.nn.functional.scaled_dot_product_attention(q,k.repeat_interleave(4,dim=1),v.repeat_interleave(4,dim=1),is_causal=True)\n    error=float((reference-candidate).abs().max())\n    if error>.02:raise RuntimeError(f\'Unexpected kernel equivalence error: {error}\')\n    sdpa_integration.use_gqa_in_sdpa=lambda attention_mask,key:False\n    AVB_SDPA_COMPAT=dict(explicit_kv_repeat=True,toy_max_abs_error=error,\n                        media_reduced=False,attention_masks_modified=False)\n    (OUT/\'avbind_sdpa_compat.json\').write_text(json.dumps(AVB_SDPA_COMPAT),encoding=\'utf-8\')\n    print(\'T4_SDPA_COMPAT\',json.dumps(AVB_SDPA_COMPAT),flush=True)\n    del q,k,v,reference,candidate;torch.cuda.empty_cache()\n', 'chronodesmear_proposal_original.py': '"""\nROUND 5 — CVPR 2027: Causal Media De-Smearing & Temporally-Stratified Counterfactual Deliberation (ChronoDeSmear)\nAuthors: Yoshua Bengio & Yann LeCun\n\nBuilt directly on the 50-configuration Round 4 Post-Mortem & Yann LeCun\'s 7 Technical Critiques:\n\n1. MATHEMATICAL ROOT-CAUSE DISCOVERY FROM ROUND 4 (`M0`, `C2`, `C3`, `D1/D2`):\n   - Why Native `M0` gets `0/5` on `TF/Past` and over-predicts `Future` (`4/5`):\n     In a causal Transformer, visual/audio tokens at timestamp `t` attend to all earlier tokens `t\' < t`.\n     Thus, an early ("Past") event leaves a persistent forward echo in "Present" and "Future" tokens,\n     while static background features (`cos > 0.92`) dominate every frame.\n   - Why `C2` (`0.5*(h_base + h_sync) - h_anti`) achieved a PERFECT `15/15 (100%)` on `CMTB` and `4/5` on `TF/Past`,\n     yet collapsed `TF/Future` to `0/5` and `TA/Yes` to `1/5`:\n     Expanding the first-order contribution of each component in `C2`:\n       `h_base = f(V_in, V_out, A)`, `h_sync = f(V_in)`, `h_anti = f(V_out, A)`\n       => `0.5*h_base + 0.5*h_sync - 1.0*h_anti ≈ +1.0 * V_in - 0.5 * V_out - 0.5 * A`\n     Because `h_anti` contained `V_out` (both Past & Future visual tokens) and `A` (Audio), subtracting `h_anti`:\n       (a) Canceled the static visual background `V_bar` (`1.0*V_bar - 1.0*V_bar = 0`) and audio shortcut -> `CMTB = 15/15`!\n       (b) Actively SUBTRACTED out-of-window visual events (`-0.5 * V_out`) -> crushed `TF/Future` to `0/5`!\n       (c) Actively SUBTRACTED Audio (`-0.5 * A`) -> crushed `TA/Yes` to `1/5`!\n   - Why `C3` (`h_base - h_aud_only`) kept `TA = 8/10` (`5/5 Yes, 3/5 No`) but only reached `8/15` on `CMTB` and collapsed `TF` to `15/15 Present`:\n     Because `h_aud_only` has NO visual tokens, `h_base - h_aud_only` failed to cancel the static visual background `V_bar`!\n\n2. THE UNIFIED SOLUTION (ZERO `speech_window` VAD, ZERO Label Words, ZERO Task Router):\n   - Mechanism A (`CausalDeSmear`): In-stream causal high-pass innovation filtering on BOTH Visual (`is_v`) and\n     Audio (`is_a`) tokens at mid-layers (`L8, 10, 12, 14, 16`). For each media token at time `t`, we subtract\n     a fraction `lambda_echo` of the mean of strictly earlier tokens (`t\' < t - 2.0s`) and restore token norm.\n     This stops Past events from echoing into Present/Future tokens in a SINGLE unmasked pass!\n   - Mechanism B (`Temporally-Stratified Static-Background Counterfactual`, `h_base - h_static_A`):\n     Without any `speech_window`, we partition the video timeline into 5 uniform temporal bins. Within EACH bin,\n     we identify the bottom 45% least-changing visual tokens (closest to the static scene mean `V_bar` and lowest\n     frame-to-frame motion) as `V_static`.\n     In the counterfactual pass `h_static_A`, only `V_static` (uniformly covering Past, Present, and Future!) and\n     Audio `A` are visible, while dynamic event tokens `V_dyn` are masked.\n     Therefore:\n       `Delta_dyn = h_base(V_dyn U V_static, A) - h_static_A(V_static, A)`\n       1. Cancels static scene background `V_bar` (`1.0 - 1.0 = 0`) just like `C2`!\n       2. Cancels unimodal audio shortcut `A` (`1.0 - 1.0 = 0`) just like `C2`!\n       3. Preserves `+1.0 * V_dyn` across ALL temporal regions (Past, Present, AND Future) with ZERO window bias!\n   - Mechanism C (`Option-Centered Decoupling & Symmetric Cross-Modal Resonance`):\n     Zero-centers the counterfactual delta across the option token slice `[opt_start : L]` (or restricts injection\n     to stem + media) so no uniform prior shift can flip `Yes -> No` or `Future -> Past`.\n\n3. CLEAN ORTHOGONAL ABLATION MATRIX (`R5_C0` to `R5_C6`):\n   - `R5_C0_baseline`          : Native Qwen2.5-Omni-3B (`18/40` reference).\n   - `R5_C1_causal_desmear_1p` : Single-Pass In-Stream Causal De-Smearing on `V` & `A` tokens ONLY (1 pass, no window, no delta).\n   - `R5_C2_stratified_dyn_cd` : Temporally-Stratified Counterfactual (`h_base - h_static_A`, NO `speech_window`!) + Option-Centered Recurrence.\n   - `R5_C3_balanced_modal_cd` : Algebraically Balanced Counterfactual (`0.5*(h_base - h_static_A) + 0.5*(0.5*h_V_all + 0.5*h_sync - h_anti)`),\n                                 explicitly restoring `+0.5*V_out` to test the exact algebraic fix for `C2`.\n   - `R5_C4_stem_plus_media`   : Injects counterfactual delta strictly into Question Stem (`last_media+1 : opt_start`) +\n                                 In-Stream Causal De-Smearing on Media tokens (`is_v, is_a`), leaving Option tokens untouched.\n   - `R5_C5_chronodesmear_full`: Full Proposed CVPR Method (Window-Free Stratified Dynamic Counterfactual + In-Stream\n                                 Causal De-Smearing + Symmetric AV Resonance Thought at `L0..2`, ZERO label words, ZERO window).\n   - `R5_C6_multiseed_shuffled`: Multi-Seed Shuffled Delta Control averaged over 3 random seeds `(42, 1337, 2027)` (Yann LeCun Req #5).\n"""\n\nimport os, sys, json, time, random\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nSHARD = int(os.environ.get(\'LAB_SHARD\', globals().get(\'SHARD\', 0)))\nNSHARD = int(os.environ.get(\'LAB_NSHARD\', globals().get(\'NSHARD\', 2)))\nPER_CELL_TOTAL = int(os.environ.get(\'LAB_PER_CELL_TOTAL\', 5))  # 5 per cell * 8 cells = 40 samples\nMAX_DUR = float(os.environ.get(\'LAB_MAX_DUR\', 50.0))\nSEED = 2027\nBS = chr(92)\n\nMETHODS = [\n    \'R5_C0_baseline\',\n    \'R5_C1_causal_desmear_1p\',\n    \'R5_C2_stratified_dyn_cd\',\n    \'R5_C3_balanced_modal_cd\',\n    \'R5_C4_stem_plus_media\',\n    \'R5_C5_chronodesmear_full\',\n    \'R5_C6_multiseed_shuffled\',\n]\n\n\ndef hidden_to_manifold_thought(h_vec, top_m=48, temp=0.65):\n    """\n    Projects a Layer-22 hidden state vector through `MODEL_.model.norm` and `MODEL_.lm_head`\n    into the Layer-0 token embedding manifold (100% free of hand-crafted label words!).\n    """\n    with torch.no_grad():\n        norm_mod = MODEL_.model.norm\n        lm_head = MODEL_.lm_head\n        dev_n = next(norm_mod.parameters()).device\n        dt_n = next(norm_mod.parameters()).dtype\n        h_in = h_vec.to(device=dev_n, dtype=dt_n).unsqueeze(0)\n        normed = norm_mod(h_in)\n        logits = lm_head(normed.to(next(lm_head.parameters()).device))[0].float()\n\n        emb_w = MODEL_.get_input_embeddings().weight.detach()\n        vals, idxs = torch.topk(logits, k=min(top_m, logits.shape[-1]))\n        probs = F.softmax(vals / temp, dim=-1).to(emb_w.device, dtype=torch.float32)\n        top_embs = emb_w[idxs.to(emb_w.device)].float()\n        c_vec = (probs.unsqueeze(-1) * top_embs).sum(dim=0).cpu()\n    return F.normalize(c_vec, dim=-1)\n\n\ndef find_stem_end_idx(stem, full_rendered_prompt, last_media, L):\n    """\n    Locates the exact token boundary `opt_start` where the Question Stem ends\n    and the Option Choices `(A) ... (B) ...` begin.\n    """\n    n_full = len(PROC.tokenizer.encode(full_rendered_prompt, add_special_tokens=False))\n    n_stem = len(PROC.tokenizer.encode(stem, add_special_tokens=False))\n    n_opts = max(4, n_full - n_stem)\n    opt_start = max(last_media + 2, min(L - 2, L - n_opts))\n    return opt_start\n\n\ndef detect_active_window(raw_audio, dur):\n    """Used ONLY for `R5_C3` to test the exact algebraic fix of C2."""\n    w = speech_window(raw_audio)\n    if w is None or (w[1] - w[0]) < 0.4:\n        return (round(0.25 * dur, 2), round(0.75 * dur, 2))\n    return (max(0.0, round(w[0] - 0.30, 2)), min(dur, round(w[1] + 0.30, 2)))\n\n\ndef select_stratified_static_visual_mask(h_base_22, is_v, sec, dur, n_bins=5, static_frac=0.45):\n    """\n    100% Window-Free, Temporally-Stratified Static Visual Selection:\n    Partitions the video timeline `[0, dur]` into `n_bins` uniform temporal intervals.\n    Within EACH interval, computes each visual token\'s Dynamic Event Innovation:\n      `I(i) = 0.55 * (1 - cos(h_v[i], h_v_mean)) + 0.45 * (1 - cos(h_v[i], h_v_prev_chunk[i]))`\n    and selects the bottom `static_frac` (45%) least-changing tokens as `is_v_static`.\n    Guarantees:\n      1. Uniform temporal coverage across Past, Present, and Future (ZERO temporal bias!).\n      2. Captures the exact static scene background `V_bar` of the video.\n    """\n    v_cpu = is_v.cpu().bool()\n    sec_cpu = sec.cpu().float()\n    L = v_cpu.shape[0]\n    is_v_static = torch.zeros(L, dtype=torch.bool)\n\n    v_idx = torch.nonzero(v_cpu, as_tuple=False).squeeze(-1)\n    if v_idx.numel() == 0:\n        return is_v_static\n\n    hv = F.normalize(h_base_22[v_idx].float(), dim=-1)  # (N_v, D)\n    hv_mean = F.normalize(hv.mean(0, keepdim=True), dim=-1)\n    dev_global = 1.0 - (hv * hv_mean).sum(dim=-1)  # (N_v,)\n\n    # Local temporal change relative to tokens ~1.5-2.0s earlier\n    v_times = sec_cpu[v_idx]\n    dev_local = torch.zeros_like(dev_global)\n    for k_i in range(v_idx.numel()):\n        t_curr = v_times[k_i]\n        prev_mask = (v_times < t_curr - 0.5) & (v_times >= t_curr - 3.0)\n        if prev_mask.any():\n            hv_prev = F.normalize(hv[prev_mask].mean(0, keepdim=True), dim=-1).squeeze(0)\n            dev_local[k_i] = 1.0 - torch.dot(hv[k_i], hv_prev)\n        else:\n            dev_local[k_i] = dev_global[k_i]\n\n    innov_score = 0.55 * dev_global + 0.45 * dev_local  # higher = more dynamic event\n\n    t_min, t_max = float(v_times.min()), float(v_times.max())\n    span = max(1e-3, t_max - t_min)\n    for b in range(n_bins):\n        b0 = t_min + (b / n_bins) * span\n        b1 = t_min + ((b + 1) / n_bins) * span + (1e-4 if b == n_bins - 1 else 0.0)\n        in_bin = torch.nonzero((v_times >= b0) & (v_times < b1), as_tuple=False).squeeze(-1)\n        if in_bin.numel() == 0:\n            continue\n        k_static = max(1, int(round(static_frac * in_bin.numel())))\n        bin_scores = innov_score[in_bin]\n        _, cleanest_local_idx = torch.topk(bin_scores, k=k_static, largest=False)\n        chosen_global_idx = v_idx[in_bin[cleanest_local_idx]]\n        is_v_static[chosen_global_idx] = True\n\n    return is_v_static, innov_score, v_idx\n\n\ndef precompute_causal_echo_weights(is_v, is_a, sec, min_lag=1.8, max_lag=12.0):\n    """\n    Precomputes sparse row-normalized causal echo matrices `W_v` and `W_a` on CPU/GPU\n    so that for any media token at timestamp `t_i`, `(W_m @ h_m)[i]` equals the mean of\n    same-modality tokens at strictly earlier timestamps `t_j in [t_i - max_lag, t_i - min_lag]`.\n    Subtracting `lambda_echo * (W_m @ h_m)` removes forward causal smearing (Past -> Future echo)!\n    """\n    sec_c = sec.cpu().float()\n    v_idx = torch.nonzero(is_v.cpu().bool(), as_tuple=False).squeeze(-1)\n    a_idx = torch.nonzero(is_a.cpu().bool(), as_tuple=False).squeeze(-1)\n\n    def build_echo_mat(idx_list):\n        n = idx_list.numel()\n        if n == 0:\n            return idx_list, torch.zeros((0, 0), dtype=torch.float32), torch.zeros(0, dtype=torch.bool)\n        t_vec = sec_c[idx_list]\n        dt = t_vec.unsqueeze(1) - t_vec.unsqueeze(0)  # dt[i, j] = t_i - t_j\n        valid = (dt >= min_lag) & (dt <= max_lag)\n        has_prev = valid.any(dim=1)\n        w = valid.float()\n        row_sum = w.sum(dim=1, keepdim=True).clamp(min=1.0)\n        w = w / row_sum\n        return idx_list, w, has_prev\n\n    v_info = build_echo_mat(v_idx)\n    a_info = build_echo_mat(a_idx)\n    return v_info, a_info\n\n\ndef make_counterfactual_mask(L, is_v, is_a, sec, dtype, device, mode=\'static_plus_aud\',\n                             is_v_static=None, win=None, spg=None):\n    """\n    Memory-safe in-place float16 4D causal mask builder:\n      - \'static_plus_aud\': Masks out dynamic visual tokens `V_dyn = V \\\\ V_static`, keeping\n                           temporally-stratified static visual background `V_static` + Audio `A`!\n      - \'v_all_only\'     : Masks out Audio `A`, keeping 100% of Visual tokens `V_all` (Past, Present, Future)!\n      - \'hard_sync\'      : C2 reference in-window visual pass.\n      - \'anti_sync\'      : C2 reference out-of-window visual + audio pass.\n    """\n    neg = torch.finfo(dtype).min\n    m = torch.zeros((1, 1, L, L), dtype=dtype, device=device)\n    m2d = m[0, 0]\n\n    triu_idx = torch.triu_indices(L, L, offset=1, device=device)\n    m2d[triu_idx[0], triu_idx[1]] = neg\n    del triu_idx\n\n    v = is_v.to(device)\n    a = is_a.to(device)\n\n    if mode == \'static_plus_aud\':\n        v_stat = is_v_static.to(device)\n        v_dyn = v & (~v_stat)\n        # Block non-dynamic tokens from attending to dynamic visual tokens `v_dyn`\n        m2d[(~v_dyn).unsqueeze(1) & v_dyn.unsqueeze(0)] = neg\n        return m\n\n    if mode == \'v_all_only\':\n        # Block non-audio tokens from attending to audio tokens `a` (keeps ALL visual tokens!)\n        m2d[(~a).unsqueeze(1) & a.unsqueeze(0)] = neg\n        return m\n\n    sec_d = sec.to(device, dtype=torch.float32)\n    s0, s1 = win\n    width = float(spg) if spg is not None else 0.0\n    inside = (sec_d + width > s0) & (sec_d < s1)\n    in_v = v & inside\n    if not in_v.any():\n        in_v = v\n    out_v = v & (~in_v)\n\n    if mode == \'hard_sync\':\n        m2d[(~out_v).unsqueeze(1) & out_v.unsqueeze(0)] = neg\n        m2d[(~a).unsqueeze(1) & a.unsqueeze(0)] = neg\n    elif mode == \'anti_sync\':\n        m2d[(~in_v).unsqueeze(1) & in_v.unsqueeze(0)] = neg\n    return m\n\n\ndef center_option_slice(delta_q, stem_len):\n    """\n    Decouples Question-Stem grounding from Option-Token prior bias:\n    - Leaves the Question Stem slice `delta_q[:stem_len]` intact.\n    - Mean-centers the Option Token slice `delta_q[stem_len:]` across the option token positions\n      and scales its norm by 0.55 so it cannot induce a one-sided label collapse (`Yes -> No` or `Future -> Past`),\n      while preserving relative token-specific action binding for `CMTB`!\n    """\n    out = delta_q.clone()\n    if out.shape[0] > stem_len + 1:\n        opt_slice = out[stem_len:]\n        opt_centered = opt_slice - opt_slice.mean(dim=0, keepdim=True)\n        out[stem_len:] = 0.55 * opt_centered\n    return out\n\n\ndef compute_av_resonance_and_shift_diagnostics(h_base_22, is_v, is_a, is_v_static, sec, dur):\n    """\n    100% Window-Free & Label-Free Cross-Modal Resonance & Grounding Verification:\n      1. Extracts dynamic visual tokens `V_dyn = V \\\\ V_static` and dynamic audio tokens `A_dyn`.\n      2. Projects both into the Layer-0 token manifold to form the symmetric Cross-Modal Resonance Thought `ct_res`.\n      3. Computes the Cross-Modal Temporal Alignment score across Early/Mid/Late thirds AND verifies\n         whether shifting visual timestamps relative to audio (`+shift` vs `0`) changes alignment\n         (fulfilling Yann LeCun\'s Requirement #6: explicit grounding verification!).\n    """\n    v_c = is_v.cpu().bool()\n    a_c = is_a.cpu().bool()\n    v_dyn = v_c & (~is_v_static.cpu().bool())\n    if not v_dyn.any():\n        v_dyn = v_c\n\n    hv_all = h_base_22[v_c].mean(0) if v_c.any() else torch.zeros(h_base_22.shape[-1])\n    hv_dyn = (h_base_22[v_dyn].mean(0) - 0.55 * hv_all) if v_dyn.any() else hv_all\n\n    if a_c.any():\n        ha_tokens = h_base_22[a_c].float()\n        ha_mean = ha_tokens.mean(0, keepdim=True)\n        a_dev = (1.0 - F.cosine_similarity(ha_tokens, ha_mean, dim=-1))\n        k_act = max(1, int(round(0.55 * ha_tokens.shape[0])))\n        _, top_a = torch.topk(a_dev, k=k_act, largest=True)\n        ha_dyn = ha_tokens[top_a].mean(0) - 0.50 * ha_mean.squeeze(0)\n    else:\n        ha_dyn = torch.zeros_like(hv_all)\n\n    z_v = hidden_to_manifold_thought(hv_dyn, top_m=48, temp=0.65)\n    z_a = hidden_to_manifold_thought(ha_dyn, top_m=48, temp=0.65)\n    ct_res = F.normalize(0.55 * z_v + 0.45 * z_a, dim=-1)\n\n    # Grounding verification across Early / Mid / Late thirds of the video\n    sec_c = sec.cpu().float()\n    d_third = max(1.0, dur / 3.0)\n    zone_cos = []\n    for z_i in range(3):\n        t0, t1 = z_i * d_third, (z_i + 1) * d_third + (0.1 if z_i == 2 else 0.0)\n        m_vz = v_dyn & (sec_c >= t0) & (sec_c < t1)\n        if not m_vz.any():\n            m_vz = v_c & (sec_c >= t0) & (sec_c < t1)\n        if m_vz.any():\n            zv_z = hidden_to_manifold_thought(h_base_22[m_vz].mean(0) - 0.5 * hv_all, top_m=32, temp=0.65)\n            zone_cos.append(round(float(torch.dot(zv_z, z_a)), 3))\n        else:\n            zone_cos.append(0.0)\n\n    return ct_res, zone_cos\n\n\ndef forward_r5(\n    inp, pos, mask=None,\n    span_start=0, span_end=None,\n    cont_thought_vec=None, cont_thought_layers=(0, 1, 2), cont_thought_scale=0.20,\n    recurrent_delta=None, recurrent_layers={10: 0.25, 12: 0.30, 14: 0.30, 16: 0.25},\n    causal_desmear_info=None, desmear_layers=(8, 10, 12, 14, 16), lambda_echo_v=0.26, lambda_echo_a=0.18,\n    record_layers=()\n):\n    emb = MODEL_.get_input_embeddings().weight\n    kw = {k: v for k, v in inp.items() if k != \'attention_mask\'}\n    captured, hooks = {}, []\n    layers = MODEL_.model.layers\n    L = inp[\'input_ids\'].shape[1]\n    s_end = L if span_end is None else span_end\n\n    for li in range(len(layers)):\n        layer = layers[li]\n\n        # 1. Layer-0..2 Continuous Resonance Thought Hook\n        if cont_thought_vec is not None and li in cont_thought_layers:\n            def make_ct_hook(li_curr):\n                def ct_hook(mod, args, out):\n                    h = out[0].clone()\n                    hq = h[0, span_start:s_end].float()\n                    ct = cont_thought_vec.to(h.device, torch.float32).unsqueeze(0)\n                    avg_n = hq.norm(dim=-1, keepdim=True).mean().clamp(min=1e-5)\n                    h[0, span_start:s_end] = (hq + (cont_thought_scale * avg_n) * ct).to(h.dtype)\n                    return (h,) + tuple(out[1:])\n                return ct_hook\n            hooks.append(layer.register_forward_hook(make_ct_hook(li)))\n\n        # 2. In-Stream Causal Echo Subtraction Hook on Visual & Audio Media Tokens (`CausalDeSmear`)\n        if causal_desmear_info is not None and li in desmear_layers:\n            def make_ds_hook(li_curr):\n                def ds_hook(mod, args, out):\n                    h = out[0].clone()\n                    (v_idx, w_v, has_prev_v), (a_idx, w_a, has_prev_a) = causal_desmear_info\n                    dev = h.device\n                    if v_idx.numel() > 0 and has_prev_v.any():\n                        idx_d = v_idx.to(dev)\n                        hv = h[0, idx_d].float()\n                        orig_norm = hv.norm(dim=-1, keepdim=True).clamp(min=1e-5)\n                        echo_v = torch.matmul(w_v.to(dev), hv)\n                        mask_pv = has_prev_v.to(dev).unsqueeze(-1).float()\n                        hv_clean = hv - (lambda_echo_v * mask_pv) * echo_v\n                        hv_clean = hv_clean * (orig_norm / hv_clean.norm(dim=-1, keepdim=True).clamp(min=1e-5))\n                        h[0, idx_d] = hv_clean.to(h.dtype)\n                    if a_idx.numel() > 0 and has_prev_a.any():\n                        idx_a = a_idx.to(dev)\n                        ha = h[0, idx_a].float()\n                        orig_norm_a = ha.norm(dim=-1, keepdim=True).clamp(min=1e-5)\n                        echo_a = torch.matmul(w_a.to(dev), ha)\n                        mask_pa = has_prev_a.to(dev).unsqueeze(-1).float()\n                        ha_clean = ha - (lambda_echo_a * mask_pa) * echo_a\n                        ha_clean = ha_clean * (orig_norm_a / ha_clean.norm(dim=-1, keepdim=True).clamp(min=1e-5))\n                        h[0, idx_a] = ha_clean.to(h.dtype)\n                    return (h,) + tuple(out[1:])\n                return ds_hook\n            hooks.append(layer.register_forward_hook(make_ds_hook(li)))\n\n        # 3. Counterfactual Latent Recurrence Hook (`L10, 12, 14, 16`)\n        if recurrent_delta is not None and li in recurrent_layers:\n            def make_rec_hook_fn(li_curr):\n                def rl_hook(mod, args, out):\n                    h = out[0].clone()\n                    hq = h[0, span_start:s_end].float()\n                    dq = recurrent_delta.to(h.device, torch.float32)\n                    beta = float(recurrent_layers[li_curr])\n                    norm_h = hq.norm(dim=-1, keepdim=True).clamp(min=1e-5)\n                    norm_d = dq.norm(dim=-1, keepdim=True).clamp(min=1e-5)\n                    h[0, span_start:s_end] = (hq + beta * (norm_h / norm_d) * dq).to(h.dtype)\n                    return (h,) + tuple(out[1:])\n                return rl_hook\n            hooks.append(layer.register_forward_hook(make_rec_hook_fn(li)))\n\n        if li in record_layers:\n            def make_cap_hook(li_curr):\n                def r_hook(mod, args, out):\n                    captured[li_curr] = out[0][0].detach().float().cpu()\n                    return out\n                return r_hook\n            hooks.append(layer.register_forward_hook(make_cap_hook(li)))\n\n    try:\n        with torch.no_grad():\n            if mask is None:\n                lg = MODEL_(**inp, use_audio_in_video=True).logits[0, -1].float().cpu()\n            else:\n                lg = MODEL_(**kw, attention_mask=mask, position_ids=pos.to(emb.device), use_audio_in_video=True).logits[0, -1].float().cpu()\n    finally:\n        for h in hooks:\n            h.remove()\n    return lg, captured\n\n\ndef make_shuffled_delta(delta_tensor, seed):\n    g = torch.Generator()\n    g.manual_seed(seed)\n    perm_idx = torch.randperm(delta_tensor.shape[-1], generator=g)\n    return delta_tensor[:, perm_idx].clone()\n\n\ndef eval_sample(r):\n    orig = fetch(r[\'video_path\'].replace(BS, \'/\'))\n    full = cut(orig, 0, 10**4, \'full\')\n    a = decode_audio(full)\n    dur = len(a) / 16000.0\n    if dur > MAX_DUR:\n        return None\n\n    base_px = 160 * 28 * 28\n    if dur > 32.0:\n        scale_factor = max(0.55, 32.0 / dur)\n        grids = max(80, int(160 * scale_factor))\n        max_px = grids * 28 * 28\n    else:\n        max_px = base_px\n\n    media = prepare_video(full, max_pixels=max_px)\n    stem, texts = parse_question(r[\'question\'])\n    k = len(texts)\n    emb = MODEL_.get_input_embeddings().weight\n    win = detect_active_window(a, dur)\n\n    totals = {m: np.zeros(k) for m in METHODS}\n    zone_cos_log = None\n\n    for perm in rotations(k):\n        rendered_q = render(stem, texts, perm)\n        inp, _ = qwen_inputs(PROC, rendered_q, media, \'av\')\n        inp = move_inputs(inp, MODEL_)\n        pos, sec, is_v, is_a, last_media = media_layout(MODEL_, PROC, inp)\n        L = inp[\'input_ids\'].shape[1]\n        spg = inp.get(\'video_second_per_grid\')\n        spg = float(spg.reshape(-1)[0]) if spg is not None else None\n\n        opt_start = find_stem_end_idx(stem, rendered_q, last_media, L)\n        stem_len = opt_start - (last_media + 1)\n\n        # Precompute Causal Echo Matrices (100% window-free, label-free!)\n        desmear_info = precompute_causal_echo_weights(is_v, is_a, sec, min_lag=1.8, max_lag=12.0)\n\n        # ======================================================================\n        # 1. `R5_C0_baseline`: Native Unmasked Pass (records Layer 22)\n        # ======================================================================\n        lg_c0, cap_base = forward_r5(inp, pos, mask=None, record_layers=(22,))\n        lp_c0 = torch.log_softmax(lg_c0[LETTERS[:k]], -1).numpy()\n        h_base_22 = cap_base[22]\n        h_base_q22 = h_base_22[last_media + 1:]\n        del cap_base\n\n        # Identify Temporally-Stratified Static Visual Tokens `is_v_static` (ZERO `speech_window`!)\n        is_v_static, _, _ = select_stratified_static_visual_mask(\n            h_base_22, is_v, sec, dur, n_bins=5, static_frac=0.45\n        )\n        ct_res, z_cos = compute_av_resonance_and_shift_diagnostics(\n            h_base_22, is_v, is_a, is_v_static, sec, dur\n        )\n        zone_cos_log = z_cos\n        del h_base_22\n\n        # ======================================================================\n        # 2. `R5_C1_causal_desmear_1p`: Single-Pass In-Stream Causal De-Smearing Only!\n        # ======================================================================\n        lg_c1, cap_ds = forward_r5(\n            inp, pos, mask=None,\n            causal_desmear_info=desmear_info, desmear_layers=(8, 10, 12, 14, 16),\n            lambda_echo_v=0.26, lambda_echo_a=0.18,\n            record_layers=(22,)\n        )\n        lp_c1 = torch.log_softmax(lg_c1[LETTERS[:k]], -1).numpy()\n        h_ds_q22 = cap_ds[22][last_media + 1:]\n        del cap_ds\n\n        # ======================================================================\n        # 3. Counterfactual Latent Extractions (`h_static_A`, `h_V_all`, `h_sync`, `h_anti`)\n        # ======================================================================\n        # 3A. Window-Free Temporally-Stratified Static Background + Audio Pass (`h_static_A`)\n        m_stat = make_counterfactual_mask(\n            L, is_v, is_a, sec, emb.dtype, emb.device, mode=\'static_plus_aud\', is_v_static=is_v_static\n        )\n        _, cap_stat = forward_r5(inp, pos, mask=m_stat, record_layers=(22,))\n        h_stat_q22 = cap_stat[22][last_media + 1:]\n        del m_stat, cap_stat\n\n        # 3B. Full-Timeline Visual-Only Pass (`h_V_all`: ALL Past/Present/Future visual tokens visible, Audio masked)\n        m_vall = make_counterfactual_mask(\n            L, is_v, is_a, sec, emb.dtype, emb.device, mode=\'v_all_only\'\n        )\n        _, cap_vall = forward_r5(inp, pos, mask=m_vall, record_layers=(22,))\n        h_vall_q22 = cap_vall[22][last_media + 1:]\n        del m_vall, cap_vall\n\n        # 3C. C2 Reference `hard_sync` & `anti_sync` (ONLY for testing the exact algebraic fix in `R5_C3`)\n        m_sync = make_counterfactual_mask(\n            L, is_v, is_a, sec, emb.dtype, emb.device, mode=\'hard_sync\', win=win, spg=spg\n        )\n        _, cap_sync = forward_r5(inp, pos, mask=m_sync, record_layers=(22,))\n        h_sync_q22 = cap_sync[22][last_media + 1:]\n        del m_sync, cap_sync\n\n        m_anti = make_counterfactual_mask(\n            L, is_v, is_a, sec, emb.dtype, emb.device, mode=\'anti_sync\', win=win, spg=spg\n        )\n        _, cap_anti = forward_r5(inp, pos, mask=m_anti, record_layers=(22,))\n        h_anti_q22 = cap_anti[22][last_media + 1:]\n        del m_anti, cap_anti\n\n        # Construct Counterfactual Deltas:\n        # (i) Window-Free Stratified Dynamic Delta: cancels static background V_bar & audio shortcut A,\n        #     preserves Past, Present, and Future dynamic events equally!\n        delta_strat_raw = (0.5 * (h_ds_q22 + h_base_q22) - h_stat_q22) + 0.35 * (h_vall_q22 - h_stat_q22)\n        delta_strat_centered = center_option_slice(delta_strat_raw, stem_len)\n\n        # (ii) Algebraically Balanced Counterfactual (`R5_C3`):\n        #      Fixes C2\'s `-0.5 * V_out - 0.5 * A` bias by restoring `h_vall_q22` (which contains `V_out`)\n        #      and option-slice mean-centering!\n        delta_c2_raw = 0.5 * (h_base_q22 + h_sync_q22) - h_anti_q22\n        delta_bal = 0.50 * delta_strat_centered + 0.50 * center_option_slice(\n            0.5 * (h_vall_q22 + h_sync_q22) - h_anti_q22, stem_len\n        )\n\n        # (iii) Pure Question-Stem Slice (`:stem_len`) for `R5_C4` (zero option-token perturbation!)\n        delta_stem_only = delta_strat_raw[:stem_len]\n\n        # ======================================================================\n        # 4. PASS 2: ORTHOGONAL ABLATION EXECUTIONS (ALL 100% UNMASKED!)\n        # ======================================================================\n        # `R5_C2_stratified_dyn_cd`: Window-Free Stratified Counterfactual + Option-Centered Recurrence\n        lg_c2, _ = forward_r5(\n            inp, pos, mask=None, span_start=last_media + 1, span_end=L,\n            recurrent_delta=delta_strat_centered,\n            recurrent_layers={10: 0.24, 12: 0.30, 14: 0.30, 16: 0.24}\n        )\n        lp_c2 = torch.log_softmax(lg_c2[LETTERS[:k]], -1).numpy()\n\n        # `R5_C3_balanced_modal_cd`: Algebraically Balanced Modal Counterfactual + In-Stream CausalDeSmear\n        lg_c3, _ = forward_r5(\n            inp, pos, mask=None, span_start=last_media + 1, span_end=L,\n            causal_desmear_info=desmear_info, desmear_layers=(8, 10, 12, 14, 16),\n            recurrent_delta=delta_bal,\n            recurrent_layers={10: 0.24, 12: 0.30, 14: 0.30, 16: 0.24}\n        )\n        lp_c3 = torch.log_softmax(lg_c3[LETTERS[:k]], -1).numpy()\n\n        # `R5_C4_stem_plus_media`: Stem-Only Counterfactual (`last_media+1 : opt_start`) + Media CausalDeSmear\n        lg_c4, _ = forward_r5(\n            inp, pos, mask=None, span_start=last_media + 1, span_end=opt_start,\n            causal_desmear_info=desmear_info, desmear_layers=(8, 10, 12, 14, 16),\n            recurrent_delta=delta_stem_only,\n            recurrent_layers={10: 0.28, 12: 0.35, 14: 0.35, 16: 0.28}\n        )\n        lp_c4 = torch.log_softmax(lg_c4[LETTERS[:k]], -1).numpy()\n\n        # `R5_C5_chronodesmear_full`: Full Window-Free ChronoDeSmear (CausalDeSmear + Stratified Dynamic CD + AV Resonance Thought)\n        lg_c5, _ = forward_r5(\n            inp, pos, mask=None, span_start=last_media + 1, span_end=L,\n            cont_thought_vec=ct_res, cont_thought_layers=(0, 1, 2), cont_thought_scale=0.18,\n            causal_desmear_info=desmear_info, desmear_layers=(8, 10, 12, 14, 16),\n            recurrent_delta=delta_strat_centered,\n            recurrent_layers={10: 0.24, 12: 0.30, 14: 0.30, 16: 0.24}\n        )\n        lp_c5 = torch.log_softmax(lg_c5[LETTERS[:k]], -1).numpy()\n\n        # `R5_C6_multiseed_shuffled`: Multi-Seed Shuffled Feature Control (3 seeds: 42, 1337, 2027)\n        lp_shuf_list = []\n        for s_seed in (42, 1337, 2027):\n            d_shuf = make_shuffled_delta(delta_strat_centered, seed=s_seed)\n            lg_shuf, _ = forward_r5(\n                inp, pos, mask=None, span_start=last_media + 1, span_end=L,\n                recurrent_delta=d_shuf,\n                recurrent_layers={10: 0.24, 12: 0.30, 14: 0.30, 16: 0.24}\n            )\n            lp_shuf_list.append(torch.log_softmax(lg_shuf[LETTERS[:k]], -1).numpy())\n        lp_c6 = np.mean(lp_shuf_list, axis=0)\n\n        scores_perm = {\n            \'R5_C0_baseline\': lp_c0,\n            \'R5_C1_causal_desmear_1p\': lp_c1,\n            \'R5_C2_stratified_dyn_cd\': lp_c2,\n            \'R5_C3_balanced_modal_cd\': lp_c3,\n            \'R5_C4_stem_plus_media\': lp_c4,\n            \'R5_C5_chronodesmear_full\': lp_c5,\n            \'R5_C6_multiseed_shuffled\': lp_c6,\n        }\n        for m, lp in scores_perm.items():\n            for pos_idx, opt_idx in enumerate(perm):\n                totals[m][opt_idx] += lp[pos_idx]\n\n        del inp\n        torch.cuda.empty_cache()\n\n    return {\n        \'dur\': round(dur, 2),\n        \'av_resonance_EML\': zone_cos_log,\n        **{m: totals[m].round(3).tolist() for m in METHODS}\n    }\n\n\ndef run_shard():\n    rows = json.loads(fetch(\'samples.json\').read_text(encoding=\'utf-8\'))\n    cells = {}\n    for i, r in enumerate(rows):\n        if r.get(\'data_type\') == \'temporal\':\n            cells.setdefault((r[\'task_type_abbr\'], r[\'answer\']), []).append((i, r))\n\n    out_file = OUT / f\'round5_chronodesmear_s{SHARD}of{NSHARD}.jsonl\'\n    if out_file.exists():\n        out_file.unlink()\n\n    records = []\n    for key in sorted(cells):\n        pool = list(cells[key])\n        random.Random(SEED).shuffle(pool)\n        my_targets = [c_idx for c_idx in range(PER_CELL_TOTAL) if (c_idx % NSHARD) == SHARD]\n        target_count = len(my_targets)\n        my_slice = pool[SHARD::NSHARD]\n        got = 0\n        for idx, r in my_slice:\n            if got >= target_count:\n                break\n            t0 = time.time()\n            try:\n                res = eval_sample(r)\n                if res is None:\n                    continue\n                gold = \'ABC\'.index(r[\'answer\'])\n                rec = {\n                    \'idx\': idx,\n                    \'task\': key[0],\n                    \'ans\': key[1],\n                    \'gold\': gold,\n                    **res,\n                    \'sec\': round(time.time() - t0, 1)\n                }\n                records.append(rec)\n                with out_file.open(\'a\', encoding=\'utf-8\') as f:\n                    f.write(json.dumps(rec) + \'\\n\')\n                got += 1\n                hits = {m: (\'Y\' if int(np.argmax(rec[m])) == gold else \'-\') for m in METHODS}\n                print(\n                    f"[{len(records)}] {key[0]}-{key[1]} idx={idx} ({rec[\'dur\']}s, res={rec[\'av_resonance_EML\']}, {rec[\'sec\']}s): " +\n                    " ".join(f"{m.split(\'_\')[1]}:{hits[m]}" for m in METHODS),\n                    flush=True\n                )\n            except Exception as e:\n                print(f"Skip idx={idx} err={repr(e)[:140]}", flush=True)\n                torch.cuda.empty_cache()\n\n    summary = {}\n    for m in METHODS:\n        by_task = {}\n        for t in (\'TA\', \'TF\', \'CMTB\'):\n            sub = [r for r in records if r[\'task\'] == t]\n            by_task[t] = round(100.0 * sum(int(np.argmax(r[m])) == r[\'gold\'] for r in sub) / max(1, len(sub)), 1)\n        by_cell = {}\n        for key in sorted(cells):\n            sub_c = [r for r in records if r[\'task\'] == key[0] and r[\'ans\'] == key[1]]\n            c_ok = sum(int(np.argmax(r[m])) == r[\'gold\'] for r in sub_c)\n            by_cell[f"{key[0]}/{key[1]}"] = f"{c_ok}/{len(sub_c)}"\n        tf_sub = [r for r in records if r[\'task\'] == \'TF\']\n        tf_pred_counts = [sum(int(np.argmax(r[m])) == c_i for r in tf_sub) for c_i in range(3)]\n        overall = round(100.0 * sum(int(np.argmax(r[m])) == r[\'gold\'] for r in records) / max(1, len(records)), 1)\n        summary[m] = {\n            **by_task,\n            \'overall\': overall,\n            \'tf_pred_ABC\': tf_pred_counts,\n            \'cells\': by_cell,\n            \'n\': len(records)\n        }\n    print("ROUND 5 CHRONODESMEAR SUMMARY:", json.dumps(summary, indent=2), flush=True)\n    return {\'shard\': SHARD, \'nshard\': NSHARD, \'summary\': summary, \'records\': records}\n\n\nRESULT = run_shard()', 'semantic_manifest_job.py': '"""Server job: build a balanced SVHalluc SEMANTIC manifest (GSA / FGSA / CSMB, all Yes/No) for the ChronoDeSmear evaluation.\nRuns inside the kaggle_rootcause namespace (fetch, cut, decode_audio, SR, BS come from there). Writes OUT/uploads/semantic_manifest.json\nwith the same schema as the frozen temporal manifest used by chronodesmear40 (total, selected, server1, server2, manifest_sha256).\nCells: (task, gold) for task in GSA, FGSA, CSMB and gold in A (Yes), B (No)  -> 6 cells, SEM_PER_CELL samples each, clip <= SEM_MAXDUR s.\nSelection uses only the seed and the clip duration, never the model or the labels\' difficulty.  Parameters (globals): SEM_PER_CELL, SEM_SEED, SEM_MAXDUR."""\nimport hashlib, json, random\n\nSEM_PER_CELL = int(globals().get(\'SEM_PER_CELL\', 10))\nSEM_SEED = int(globals().get(\'SEM_SEED\', 31337))\nSEM_MAXDUR = float(globals().get(\'SEM_MAXDUR\', 40.0))\nrows = json.loads(fetch(\'samples.json\').read_text(encoding=\'utf-8\'))\ncells = {}\nfor i, r in enumerate(rows):\n    if r[\'data_type\'] == \'semantic\': cells.setdefault((r[\'task_type_abbr\'], r[\'answer\']), []).append(i)\n\nselected = []\nfor key in sorted(cells):\n    ids = cells[key][:]; random.Random(f\'{SEM_SEED}-{key}\').shuffle(ids); got = []; tried = 0\n    for i in ids:\n        r = rows[i]; tried += 1\n        try:\n            path = r[\'video_path\'].replace(BS, \'/\')\n            dur = len(decode_audio(cut(fetch(path), 0, 10 ** 4, \'full\'))) / SR\n        except Exception as e:\n            print(\'skip\', i, repr(e)[:80], flush=True); continue\n        if dur > SEM_MAXDUR or dur < 4.0: continue\n        got.append(dict(idx=i, task=key[0], gold=key[1], video_path=path, question=r[\'question\'],\n                        duration=round(dur, 3), decoded_audio_duration=round(dur, 3)))\n        if len(got) == SEM_PER_CELL: break\n    print(key, \'selected\', len(got), \'after\', tried, \'tried\', flush=True)\n    selected.extend(got)\n\n# alternate samples between the two servers inside every cell, so both servers see all cells\nserver1, server2 = [], []\nfor key in sorted(cells):\n    cell = [s for s in selected if (s[\'task\'], s[\'gold\']) == key]\n    server1.extend(cell[0::2]); server2.extend(cell[1::2])\nmanifest = dict(total=len(selected), per_cell=SEM_PER_CELL, seed=SEM_SEED, max_duration=SEM_MAXDUR,\n                selected=selected, server1=server1, server2=server2)\nmanifest[\'manifest_sha256\'] = hashlib.sha256(json.dumps(selected, sort_keys=True).encode()).hexdigest()\n(OUT / \'uploads\').mkdir(exist_ok=True)\n(OUT / \'uploads\' / \'semantic_manifest.json\').write_text(json.dumps(manifest), encoding=\'utf-8\')\n(OUT / \'semantic_manifest.json\').write_text(json.dumps(manifest, indent=1), encoding=\'utf-8\')\nRESULT = dict(total=len(selected), server1=len(server1), server2=len(server2), sha=manifest[\'manifest_sha256\'])\nprint(RESULT, flush=True)\n'}
for _n, _s in SRC.items(): (ROOT/_n).write_text(_s, encoding='utf-8')
print('Đã ghi:', list(SRC), '| params', PARAMS)


In [ ]:
# CELL 3 — chạy thử 2 mẫu (dừng nếu lỗi)
import subprocess, json as _json
def run_job(out_dir, params):
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(PYTHON), str(ROOT/'nb_runner.py'), str(ROOT), str(out_dir), str(ROOT/JOB), _json.dumps(params)]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        low = line.lower()
        if ('warn' in low or 'loading checkpoint' in low or 'fetching' in low or 'it/s]' in low) and 'traceback' not in low: continue
        print(line, end='', flush=True)
    return p.wait()
SMOKE = OUT/'smoke'
code = run_job(SMOKE, {**PARAMS, 'LIMIT': 4, 'HOURS': 1.0})
rows = [_json.loads(l) for f in SMOKE.glob('*.jsonl') for l in f.read_text().splitlines() if l.strip()]
good = [r for r in rows if 'error' not in r]
real_err = [r for r in rows if 'error' in r and 'long' not in r['error'] and 'no room' not in r['error']]   # 'long' = clip over the length limit, not a bug
print('\nSMOKE: exit', code, '| items', len(rows), '| ok', len(good), '| real errors', len(real_err))
for r in rows:
    if 'error' in r: print('  error:', r['error'][:200])
if code != 0 or real_err:
    raise RuntimeError('Smoke test thất bại: dừng để không phí GPU. Gửi log của cell này cho Claude.')
print('Smoke test OK -> chạy job đầy đủ.' if good else 'Cả 4 clip thử đều quá dài (không phải lỗi) -> vẫn chạy job đầy đủ.')


In [ ]:
# CELL 4 — chạy job đầy đủ (tự dừng trước hạn)
import subprocess, json as _json
def run_job(out_dir, params):
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(PYTHON), str(ROOT/'nb_runner.py'), str(ROOT), str(out_dir), str(ROOT/JOB), _json.dumps(params)]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        low = line.lower()
        if ('warn' in low or 'loading checkpoint' in low or 'fetching' in low or 'it/s]' in low) and 'traceback' not in low: continue
        print(line, end='', flush=True)
    return p.wait()
code = run_job(OUT/'full', PARAMS)
print('exit code', code)


In [ ]:
# CELL 5 — tóm tắt
for f in (OUT/"full").glob("*.jsonl"):
    rows = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
    print(f.name, "| items", len(rows), "| errors", sum("error" in r for r in rows))


In [ ]:
# CELL 6 — nén kết quả (jsonl + npz + log; không kèm video)
ZIP = Path(f'/kaggle/working/{ZIPNAME}.zip')
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in (OUT/'full').rglob('*'):
        if f.is_file() and f.suffix in ('.jsonl', '.json', '.npz', '.log', '.pt') and 'clips' not in f.parts:
            z.write(f, str(f.relative_to(OUT/'full')))
    for f in (OUT/'smoke').glob('*.jsonl'): z.write(f, 'smoke/' + f.name)
print(ZIP, round(ZIP.stat().st_size / 1e6, 1), 'MB')
display(FileLink(str(ZIP)))
